## About this notebook

This notebook is the record of the search development: every measurement, with its code and files. The [search strategy](../../materials/search-strategy.qmd) page summarizes the method, the results and their limits, and `log.md` records each decision with its date.

**Names used in the runs.**

- **The phrase:** *"data sharing"*, plus *"shar\* data"* in PubMed.
- **Phrase plus factor words:** the phrase in the title, OR the phrase with any of the candidate factor terms, within 5 words of it in Scopus and Web of Science.
- **Share near data:** a form of *share* within 3 or 5 words of *data*.
- **Candidate 2:** share within 5 words of data, in the title, before computing words are left out (run 11).
- **The query:** share within 5 words of data, in the title, leaving out titles that name computing systems.
- **The revised query:** share within 5 words of data AND (research\* OR scienti\*), in titles and abstracts, with no NOT clause (runs 25 to 31).

Runs 1 to 15 count development sources out of 40 with a DOI or PMID, because they ran before sources about reuse alone left the known sources; the development set now has 39. Runs 1 to 21 ran on 18 November 2026, and runs 22 to 31 on 23 November 2026.

In [ ]:
import math
import re
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "search-development"
PROCESSED = ROOT / "data" / "processed" / "search-development"
PROCESSED_FILES = {"reference_sources.csv", "counts.csv", "verdict.csv", "statement_matches.csv",
                   "terms.csv", "term_syntax.csv"}
DATABASES = {"PubMed": "pubmed", "Scopus": "scopus", "Web of Science": "web-of-science"}
PARTS = ["sharing", "reuse", "factors that obstruct", "factors that facilitate", "stakeholder perspectives",
         "policies and what drive them", "mechanisms for implementation"]


def path(name):
    return (PROCESSED if name in PROCESSED_FILES else RAW) / name


def read(name):
    return pd.read_csv(path(name), dtype=str, keep_default_na=False)



In [ ]:
#| output: false
import re

CODES = "reference_codes_qwen3.5-122b.csv"
THEMES = ["factors that obstruct", "factors that facilitate", "stakeholder perspectives",
          "policies and what drive them", "mechanisms for implementation"]
STATEMENTS = [("research data sharing", "researchers"), ("digital curation", "data sharing"),
              ("research data sharing", "open science"), ("research data sharing", "open data")]

cand = read("reference_candidates.csv")
look = read("reference_lookups.csv")
codes = read(CODES)


def write(df, name):
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED / name, index=False, lineterminator="\n")


# one value per source for the single-value fields
single = (codes[codes["field"].isin(["practice", "data_origin", "field", "source_type", "language"])]
          .pivot(index="ref_id", columns="field", values="value"))
themes_of = codes[(codes["field"] == "factors") & (codes["value"] != "none")].groupby("ref_id")["theme"].agg(set)

src = look[["ref_id", "title", "pub_date", "doi", "pmid", "abstract_source"]].merge(
    cand[["ref_id", "label", "cited_in"]], on="ref_id").join(single, on="ref_id")
src["year"] = src["pub_date"].str[:4]
src["set"] = src["year"].astype(int).map(lambda y: "development" if y <= 2014 else "test")
src["tier"] = src["field"].map(lambda f: "1" if f == "health" else "2")
src["coded_on"] = src["ref_id"].map(codes.groupby("ref_id")["coded_on"].first())
src["themes"] = src["ref_id"].map(lambda r: "; ".join(sorted(themes_of.get(r, set()))))


def reasons(s, scope):
    """The codebook.yml exclusion codes a source meets, in the codebook's order. Year and tier count
    only for In scope, not for Eligible apart from year."""
    out = []
    if s["practice"] in ("neither", "reuse"):  # the review asks about sharing; reuse alone falls outside it
        out.append("not-data-sharing")
    if not s["themes"]:
        out.append("not-factors")
    if s["data_origin"] == "not-research-data":
        out.append("not-research-data")
    if s["language"] not in ("English", "French"):
        out.append("language")
    if scope and not 2015 <= int(s["year"]) <= 2025:
        out.append("date")
    return out


src["all_reasons"] = src.apply(lambda s: ";".join(reasons(s, True)), axis=1)
src["reason"] = src["all_reasons"].str.split(";").str[0]
src["in_scope"] = src["all_reasons"].map(lambda r: "no" if r else "yes")
src["eligible_apart_from_year"] = src.apply(lambda s: "no" if reasons(s, False) else "yes", axis=1)
src = src[["ref_id", "label", "title", "year", "pub_date", "doi", "pmid", "cited_in", "set", "field", "tier",
           "source_type", "practice", "data_origin", "themes", "language", "coded_on", "abstract_source",
           "eligible_apart_from_year", "in_scope", "reason", "all_reasons"]].sort_values("ref_id")
write(src, "reference_sources.csv")

# counts: sources with each value of each coded field, by set, and codes resting on no passage
setof = src.set_index("ref_id")["set"]
c = codes.assign(set=codes["ref_id"].map(setof), nopassage=codes["passage"].eq("none") & codes["field"].ne("language"))
rows = []
for (field, value, theme), g in c.groupby(["field", "value", "theme"]):
    per = g.groupby("ref_id").agg(set=("set", "first"), nopassage=("nopassage", "all"))
    rows.append(dict(field=field, value=value, theme=theme,
                     development=int((per["set"] == "development").sum()), test=int((per["set"] == "test").sum()),
                     total=len(per), codes_without_passage=int(g["nopassage"].sum())))
for field in ("tier", "eligible_apart_from_year", "in_scope", "reason"):
    for value, g in src.groupby(field):
        rows.append(dict(field=field, value=value or "(none)", theme="",
                         development=int((g["set"] == "development").sum()), test=int((g["set"] == "test").sum()),
                         total=len(g), codes_without_passage=""))
write(pd.DataFrame(rows), "counts.csv")

# verdict: sources eligible apart from year coded for each PCC part, by set
elig = src[src["eligible_apart_from_year"] == "yes"]
has = {"sharing": elig["practice"].isin(["sharing", "both"]), "reuse": elig["practice"].isin(["reuse", "both"])}
has.update({t: elig["themes"].str.contains(t, regex=False) for t in THEMES})
vrows = []
for part, mask in has.items():
    d, t = int((mask & (elig["set"] == "development")).sum()), int((mask & (elig["set"] == "test")).sum())
    verdict = "covered" if d and t else "development only" if d else "test only" if t else "not covered"
    vrows.append(dict(pcc_part=part, development=d, test=t, verdict=verdict))
write(pd.DataFrame(vrows), "verdict.csv")

# Chawinga and Zinn's search statements against titles and abstracts, as phrases and as separate words
squash = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", s.lower()).strip() + " "
text = look.set_index("ref_id").apply(lambda r: squash(r["title"] + " " + r["abstract"]), axis=1)
inscope = set(src.loc[src["in_scope"] == "yes", "ref_id"])
srows = []
for a, b in STATEMENTS:
    for reading in ("phrase", "words"):
        if reading == "phrase":
            hit = lambda t: all(squash(x) in t for x in (a, b))
        else:
            hit = lambda t: all(f" {w} " in t for x in (a, b) for w in squash(x).split())
        ids = sorted(r for r, t in text.items() if hit(t))
        srows.append(dict(statement=f"{a} AND {b}", reading=reading, matched=len(ids),
                          matched_in_scope=sum(r in inscope for r in ids), ref_ids=";".join(ids)))
write(pd.DataFrame(srows), "statement_matches.csv")
print(f"Built reference_sources.csv ({len(src)} sources), counts.csv, verdict.csv and statement_matches.csv.")

# Step 2's terms: recorded terms from the codes, and MeSH headings, author keywords and OpenAlex keywords
# placed under a PCC part by the assignments of record.
ASSIGNMENTS = "term_assignments_qwen3.5-122b.csv"
DBS = ["PubMed", "Scopus", "Web of Science"]
FIELD = {"PubMed": "[tiab]", "Scopus": "TITLE-ABS-KEY", "Web of Science": "TS"}
# Fallback terms from the candidate terms (listed under Phrase and factor terms in this notebook). Reuse always takes its own:
# nearly every development set source coded for reuse is also coded for sharing, so the yield rule can't
# judge a reuse term on sources a sharing term doesn't already reach.
FALLBACK = {"reuse": ["data reuse", "data re-use"]}

dev = src[src["set"] == "development"]
dev_ids = set(dev["ref_id"])
doi_of = src.set_index("ref_id")["doi"]
assign = read(ASSIGNMENTS)
indexing = read("reference_indexing.csv")


def norm(t):
    t = t.lower().strip().strip('"').replace("’", "'")
    return re.sub(r"\s+", " ", re.sub(r"[-‑]", " ", t)).strip()


def stem(w):
    for suffix in ("ies", "es", "s"):
        if w.endswith(suffix) and len(w) > 4:
            return w[:-len(suffix)] + ("y" if suffix == "ies" else "")
    return w


def merge_key(t):
    """Variants that differ only by hyphen, plural or word ending share a key."""
    return " ".join(stem(w) for w in norm(t).split())


def parts_of(value, theme):
    if value in ("none", "neither"):
        return []
    if value == "both":
        return ["sharing", "reuse"]
    if value in ("sharing", "reuse"):
        return [value]
    return [theme] if theme else []


# occurrences: (key, type, part) -> sources carrying it, its variants, and its assignment basis
occ = {}


def add(key, kind, part, ref_id, variant, basis):
    o = occ.setdefault((key, kind, part), {"src": set(), "variants": set(), "basis": basis})
    o["src"].add(ref_id)
    o["variants"].add(variant)


usable = codes[codes["ref_id"].isin(dev_ids) & codes["field"].isin(["practice", "factors"])
               & ~codes["term"].isin(["", "none"]) & (codes["problem"] == "")]
for _, c in usable.iterrows():
    for part in parts_of(c["value"], c["theme"]):
        add(merge_key(c["term"]), "recorded", part, c["ref_id"], norm(c["term"]), "recorded")
value_of = {(a["type"], a["term"]): (a["value"], a["theme"]) for _, a in assign.iterrows()}
for _, i in indexing[indexing["ref_id"].isin(dev_ids)].iterrows():
    value, theme = value_of.get((i["type"], i["term"]), ("none", ""))
    for part in parts_of(value, theme):
        key = i["term"] if i["type"] == "MeSH" else merge_key(i["term"])
        add(key, i["type"], part, i["ref_id"], i["term"] if i["type"] == "MeSH" else norm(i["term"]), value)

# coded sources per PCC part, and fallback terms where a Practice value lacks its own sources
coded = {p: set() for p in PARTS}
for _, c in codes[codes["ref_id"].isin(dev_ids)].iterrows():
    if c["field"] in ("practice", "factors"):
        for part in parts_of(c["value"], c["theme"]):
            coded[part].add(c["ref_id"])
for part, terms_ in FALLBACK.items():
    occ[(terms_[0], "fallback", part)] = {"src": set(), "variants": set(terms_), "basis": "Candidate terms table"}


def written(key, variants, kind, db):
    """One database's written form: MeSH as [mh:noexp] in PubMed only; single words sharing a stem of
    5 or more letters as truncation; otherwise the variants, phrases in quotes, joined with OR."""
    if kind == "MeSH":
        return f'"{key}"[mh:noexp]' if db == "PubMed" else None
    vs = sorted(variants)
    if all(" " not in v for v in vs) and (len(vs) > 1 or stem(vs[0]) != vs[0]):
        root = min((stem(v) for v in vs), key=len)
        while not all(v.startswith(root) for v in vs):
            root = root[:-1]
        forms = [root + "*"] if len(root) >= 5 else vs
    else:
        forms = [f'"{v}"' if " " in v else v for v in vs]
    if db == "PubMed":
        return " OR ".join(f"{f}[tiab]" for f in forms)
    q = " OR ".join(forms)
    return f"TITLE-ABS-KEY({q})" if db == "Scopus" else f"TS=({q})"


# A sharing or reuse term goes to step 3 only if it names data, or is a MeSH heading the model placed by
# its scope note: the Concept is the sharing and reuse of research data, and a term such as "use" or
# "sharing" alone searches for any use or sharing.
names_data = lambda term: re.search(r"data", term.lower()) is not None
shown = lambda key, o, kind: key if kind == "MeSH" else min(o["variants"], key=lambda v: (len(v), v))
syntax_rows, term_rows = [], []
for db in DBS:
    here = {k: o for k, o in occ.items() if not (k[1] == "MeSH" and db != "PubMed")}
    seen = set()
    for (key, kind, part), o in sorted(here.items()):
        if (key, kind) not in seen:
            seen.add((key, kind))
            syntax_rows.append(dict(term=shown(key, o, kind), type=kind, database=db,
                                    variants="; ".join(sorted(o["variants"])),
                                    written_form=written(key, o["variants"], kind, db)))
    for part in PARTS:
        ranked = sorted(((k, o) for k, o in here.items() if k[2] == part),
                        key=lambda x: (-len(x[1]["src"]), x[0][0], x[0][1]))
        matched = set()
        for (key, kind, _), o in ranked:
            if part in ("sharing", "reuse") and kind != "MeSH" and not names_data(shown(key, o, kind)):
                continue
            row = dict(term=shown(key, o, kind), type=kind, database=db,
                       field="[mh:noexp]" if kind == "MeSH" else FIELD[db], pcc_part=part,
                       sources_in_part=len(coded[part]), assignment_basis=o["basis"], count=len(o["src"]))
            if part in ("sharing", "reuse"):
                row.update(sources_added="", dois=";".join(doi_of[r] or r for r in sorted(o["src"])), status="")
            else:
                added = o["src"] - matched
                if not added:
                    continue
                matched |= added
                row.update(sources_added=len(added), dois=";".join(doi_of[r] or r for r in sorted(added)),
                           status="to screening")
            term_rows.append(row)
write(pd.DataFrame(syntax_rows), "term_syntax.csv")
write(pd.DataFrame(term_rows), "terms.csv")
print(f"Built term_syntax.csv ({len(syntax_rows)} written forms) and terms.csv ({len(term_rows)} rows).")

# precision_screening.html: the three precision samples in one page for screening, from the template beside
# this notebook, with the exclusion codes from codebook.yml
import json
import yaml

records, cols = [], None
for db, slug in DATABASES.items():
    p = path(f"precision_sample_{slug}.csv")
    if not p.exists():
        continue
    s = read(p.name)
    cols = cols or list(s.columns) + ([] if "note" in s.columns else ["note"])
    for i, r in enumerate(s.to_dict("records")):
        records.append(dict(db=db, slug=slug, n=i + 1, of=len(s), id=r["record_id"], doi=r["doi"], title=r["title"],
                            year=r["year"], source=r["abstract_source"], abstract=r["abstract"],
                            decision=r.get("decision", ""), code=r.get("exclusion_code", ""), note=r.get("note", "")))
if records:
    codes_yml = yaml.safe_load((ROOT / "codebook.yml").read_text(encoding="utf-8"))["exclusion_reasons"]
    data = json.dumps(dict(records=records, codes=codes_yml, cols=cols), ensure_ascii=False).replace("</", "<\\/")
    present = [db for db in DATABASES if any(r["db"] == db for r in records)]
    template = (ROOT / "analysis" / "00_search-development" / "precision_screening_template.html").read_text(encoding="utf-8")
    page = template.replace("__DATA__", data).replace("__DBS__", ", ".join(present))
    (PROCESSED / "precision_screening.html").write_text(page, encoding="utf-8")
    print(f"Built precision_screening.html ({len(records)} records from {', '.join(present)}).")


In [ ]:
steps_1_2 = ["reference_candidates.csv", "reference_sources.csv", CODES, "counts.csv",
             "verdict.csv", "comparison.csv", "statement_matches.csv", "terms.csv", "term_syntax.csv"]
missing = [n for n in steps_1_2 if not path(n).exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))
else:
    cand, src, codes, counts, verdict, comparison, statements, terms, syntax = (read(n) for n in steps_1_2)
have_tests = path("query_tests.csv").exists()

## Known sources

Known sources are sources the query should find. [Chawinga and Zinn (2019)](https://doi.org/10.1016/j.lisr.2019.04.004) reviewed research data sharing, and the sources they included serve as known sources once checked against the RDS-Review criteria.

### Rebuilding Chawinga and Zinn's list

Chawinga and Zinn did not publish a list of their 105 included sources, and neither the paper's Crossref record nor the first author's thesis and data deposits hold one. `retrieval/reference_set.py` rebuilt the list from the paper's full text, read through Elsevier's Article Retrieval API, and keeps only bibliographic details. It takes the references Table 2 names, the references sections 4 and 5 cite, and, for each country section 4.3 lists that no other selected reference names, the reference from 2007 to 2018 whose title names it. The result is 102 sources. `retrieval/lookups.py` checked each DOI against Crossref and OpenAlex and pulled the PMID, the earliest publication date, the title, the abstract, the MeSH headings and the keywords.

Files: [reference_candidates.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_candidates.csv), [reference_lookups.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_lookups.csv).

### Coding

A local model, Qwen3.5-122B-A10B, coded each source from its title and abstract with `prompts/coding_prompt.md`, which holds the definitions. It recorded whether the source concerns sharing, reuse, both or neither, whether the data are research data, the factors the source reports under the screening guide's five themes, its field, its source type and its language, each with the sentence that supports it. A smaller model, Qwen3.8-27B, coded the same sources with the same prompt, and the two agree on 79% to 100% of sources for each variable.

Most sources concern sharing: 63 are coded `sharing` and 22 `both`. Policies (47), infrastructure (46) and barriers (31) are the factors most reported. 14 sources report no factor on the list. The codes that rest on no passage are mostly factors `none` (14), `data_origin` `unclear` (10) and `source_type` `other` (7), which mark what the title and abstract do not state. Four codes carry a problem from the coding checks. One passage is not in the title or abstract, and three carry a copied "Title:" label.

The notebook also checks that every coded source is in the reference set, that each source has exactly one code for each variable that takes one value, and that every term in `terms.csv` has a written form. A failed check stops the notebook, and all checks pass.

Files: [counts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/counts.csv), [reference_codes_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_codes_qwen3.5-122b.csv).

In [ ]:
#| output: false
if have_data:
    errors = []
    ids = set(src["ref_id"])
    if set(cand["ref_id"]) != ids:
        errors.append(f"reference_sources.csv and reference_candidates.csv list different sources: {sorted(set(cand['ref_id']) ^ ids)}.")
    unknown = set(codes["ref_id"]) - ids
    if unknown:
        errors.append(f"{CODES} has sources not in reference_sources.csv: {sorted(unknown)}.")
    for f in ("practice", "data_origin", "field", "source_type", "language"):
        n = codes[codes["field"] == f].groupby("ref_id").size().reindex(sorted(ids), fill_value=0)
        if (n != 1).any():
            errors.append(f"{int((n != 1).sum())} source(s) lack exactly one {f} code: {sorted(n[n != 1].index)}.")
    pairs = set(zip(syntax["term"], syntax["type"], syntax["database"]))
    unwritten = [t for t in zip(terms["term"], terms["type"], terms["database"]) if t not in pairs]
    if unwritten:
        errors.append(f"{len(unwritten)} row(s) in terms.csv have no written form in term_syntax.csv: {unwritten[:5]}.")
    if errors:
        raise ValueError("\n".join(errors))
    print("All checks passed.")
    flagged = codes[codes["problem"] != ""]
    if len(flagged):
        print(f"{len(flagged)} code(s) in {CODES} carry a problem from the coding checks:")
        for _, f in flagged.iterrows():
            print(f"  {f['ref_id']}, {f['field']} {f['value']}: {f['problem']}")


### Development and test sets

The notebook applies the RDS-Review criteria to the codes. A source is eligible apart from year when it concerns sharing, alone or with reuse, is not coded as concerning data from outside research, reports at least one factor, and is in English or French. Sources published 2007 to 2014 form the development set, which the runs measure candidate queries against. Sources published from 2015 form the test set, held out to measure the chosen query.

45 of the 60 development sources and 31 of the 42 test sources are eligible apart from year, and 39 and 28 of them have a DOI or PMID, so the databases can be checked for them. Of the sources that are not eligible, most fall under `not-data-sharing` (8 development, 9 test), the rest under `not-factors` (5 and 1) and `not-research-data` (2 and 1). The development set fails the date criterion by design.

A PCC part is covered when sources eligible apart from year carry it in both sets. All seven PCC parts are covered in both sets. Stakeholder perspectives rests on 6 development and 4 test sources, the fewest of any part.

Files: [reference_sources.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/reference_sources.csv), [verdict.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/verdict.csv).

### How their search compares

Chawinga and Zinn's criteria are narrower on source type, empirical design and language, and overlap with RDS-Review's on topic and date. They set no criterion on reuse, data origin, factors or field, so the codes show what their set covers.

The notebook matched the four search statements Chawinga and Zinn report against the titles and abstracts of the reference set, once with each part as a phrase and once as separate words. Read as phrases, the four statements match at most 2 sources; read as separate words, at most 29, 11 of them in scope. Chawinga and Zinn's statements reach a minority of the sources they included, and offer no template for the RDS-Review query.

Files: [comparison.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/comparison.csv), [statement_matches.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/statement_matches.csv).

## Choosing the query

Each run measures candidate queries in PubMed, Scopus and Web of Science on two counts: the records they return for 2015 to 2025 in English or French, and the development sources they find, checked by DOI or PMID with no date limit. A candidate drops out when another finds at least as many development sources in fewer records. The test set stays out of every run until run 16.

The runs query the databases' APIs, so they run on the analysis server from the repository root, with the keys loaded (`source ~/.rds-review-keys`). Rendering skips them (`eval: false`). To repeat a run, run the setup cell and then the run's cell.

In [ ]:
#| eval: false
# Setup for every run below.
import re
import sys
import itertools

sys.path.insert(0, str(ROOT / "analysis" / "00_search-development" / "retrieval"))
import search_lib as sl

sl.BATCH = 10  # identifiers per request when checking which development studies a query finds

LANG = {"PubMed": "(english[la] OR french[la])", "Scopus": "LANGUAGE(english OR french)",
        "Web of Science": "LA=(English OR French)"}
TAG = {"title": {"PubMed": "{}[ti]", "Scopus": "TITLE({})", "Web of Science": "TI=({})"},
       "anywhere": {"PubMed": "{}[tiab]", "Scopus": "TITLE-ABS-KEY({})", "Web of Science": "TS=({})"}}

# The sharing block that run 1 settled on.
SHARING = {"PubMed": ['"data sharing"', '"shar* data"'],
           "Scopus": ['"data sharing"'],
           "Web of Science": ['"data sharing"']}

# The candidate factor terms (listed under Phrase and factor terms), then general words for a factor that the list lacks.
CANDIDATE_TERMS = ["polic*", "challenge*", "repositor*", "ethic*", "privacy", "infrastructure", "barrier*",
                   "perception*", '"open science"', "motivat*", "interoperab*", '"open data"', '"open access"',
                   "consent", '"open research"', "licens*", "attitude*", "incentive*", "mandate*", "willingness",
                   "obstacle*", "stewardship", '"data management plan*"', '"data curation"', "facilitator*",
                   "enabler*", "reluctan*", '"digital curation"']
GENERAL_WORDS = ["factor*", "influenc*", "determinant*", "driver*", "concern*", "trust"]
ALL_FACTORS = CANDIDATE_TERMS + GENERAL_WORDS
ZERO_STUDY = ["facilitator*", "driver*", "interoperab*", '"open research"', '"digital curation"']  # run 5
OUT_OF_SCOPE = ["privacy"]  # run 6
FACTORS = [t for t in ALL_FACTORS if t not in ZERO_STUDY + OUT_OF_SCOPE]


def limit(db):
    return f"{sl.date_limit(db, 2015, 2025)} AND {LANG[db]}"


def field(db, where, terms):
    """The terms OR-joined in one field: the title, or anywhere in title, abstract and keywords."""
    return "(" + " OR ".join(TAG[where][db].format(t) for t in terms) + ")"


def version3(db, factors, near):
    """Data sharing in the title, OR data sharing with a factor term anywhere, or within 5 words with near."""
    v1 = field(db, "title", SHARING[db])
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if not near:
        v2 = f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)})"
    elif db == "Scopus":
        v2 = f"TITLE-ABS-KEY(({s}) W/5 ({f}))"
    else:
        v2 = f"TS=(({s}) NEAR/5 ({f}))"
    return f"({v1} OR {v2})"


FORMS = ("sharing", "shared", "share", "shares")  # PubMed proximity takes no truncation


def share_near_data(db, where, n):
    """A form of share within n words of data, in the title or anywhere. Any word order matches."""
    if db == "PubMed":
        tag = "ti" if where == "title" else "tiab"
        return "(" + " OR ".join(f'"{w} data"[{tag}:~{n}]' for w in FORMS) + ")"
    t = {"Scopus": {"title": "TITLE", "anywhere": "TITLE-ABS-KEY"},
         "Web of Science": {"title": "TI=", "anywhere": "TS="}}[db][where]
    return f"{t}(shar* W/{n} data)" if db == "Scopus" else f"{t}(shar* NEAR/{n} data)"


def limit_near(db):
    """Web of Science's API rejects LA beside shar* NEAR/n data (run 10), so there the date limit stands alone."""
    return sl.date_limit(db, 2015, 2025) if db == "Web of Science" else limit(db)


def fetch(db, q, lim):
    """Every record a query returns, with DOI, title and year."""
    full = f"({q}) AND {lim}"
    if db == "PubMed":
        return list(sl.pubmed_summary(sl.pubmed_all_ids(full)).values())
    return sl.scopus_records(full) if db == "Scopus" else sl.wos_records(full)


def unique_count(recs):
    """Records left once those sharing a DOI, or a title and year, count as one. bib-dedupe finds as many or more."""
    seen, unique = {}, 0
    for r in recs:
        keys = [k for k in (("doi", r["doi"]) if r["doi"] else None,
                            ("title", re.sub(r"[^a-z0-9]", "", r["title"].lower()) + r["year"]) if r["title"] else None)
                if k]
        hit = next((seen[k] for k in keys if k in seen), None)
        if hit is None:
            unique += 1
            hit = unique
        for k in keys:
            seen[k] = hit
    return unique


SRC = sl.read("reference_sources.csv")
DEV = [s for s in SRC if s["set"] == "development" and s["eligible_apart_from_year"] == "yes"]
LABEL = {s["ref_id"]: s["label"] for s in SRC}


def dev_in(db):
    """Development studies a database can be checked for: a DOI, or in PubMed a PMID."""
    return [s for s in DEV if sl.testable(db, s)]

### Phrase and factor terms

The candidate factor terms were drafted before search development began, from the PCC table's factor themes: policies, policy, challenge\*, repositor\*, ethic\*, privacy, infrastructure, barrier\*, perception\*, open science, motivat\*, interoperab\*, "open data", open access, consent, open research, licens\*, attitude\*, incentive\*, mandate\*, willingness, obstacle\*, stewardship, "data management plan\*", "data curation", facilitator\*, enabler\*, reluctan\* and "digital curation". "data reuse" and "data re-use" were the candidates for reuse, which the review later left out.

#### Run 1. Sharing terms

**Question.** Does any phrase for sharing data find development sources that *"data sharing"* misses, at a cost worth paying?

**Measured.** 18 candidate phrases for sharing, among them *"shar\* data"*, *"data access"*, *"data availability"* and *"data publication"*, each alone in title, abstract and keywords in the three databases: its records, the development sources it finds, how many of those *"data sharing"* misses, and how many no other phrase finds.

**Result.** In PubMed, *"shar\* data"* finds three development sources *"data sharing"* misses: Bertzky and Stoll-Kleemann (2009), Cooper (2007), and Rohlfing and Poline (2012). *"data publication"*, *"data access"* and *"data availability"* each add only Costello (2009), for 600 to 18,000 records.

**Decision.** The phrase becomes *"data sharing"*, plus *"shar\* data"* in PubMed. The other phrases stay out.

Files: [term_table_sharing.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing.csv).

In [ ]:
#| eval: false
SHARING_CANDIDATES = ['"data sharing"', '"research data sharing"', '"shar* data"', '"shared data"',
                      '"sharing of data"', '"open data sharing"', '"data access"', '"data availability"',
                      '"data publication"', '"data dissemination"', '"data deposit*"', '"data repositor*"',
                      '"open data"', '"data reuse"', '"data re-use"', '"secondary use"', '"FAIR data"',
                      '"FAIR principles"']
MESH = '"Information Dissemination"[mh:noexp]'  # PubMed only

rows = []
for db in DATABASES:
    dev = dev_in(db)
    terms = SHARING_CANDIDATES + ([MESH] if db == "PubMed" else [])
    found = {}
    for t in terms:
        w = t if t == MESH else TAG["anywhere"][db].format(t)
        found[t] = (sl.count(db, f"({w}) AND {limit(db)}"), sl.retrieved(db, w, dev))
    for t in terms:
        n, hits = found[t]
        others = set().union(*(found[o][1] for o in terms if o != t))
        rows.append(dict(database=db, term="Information Dissemination [MeSH]" if t == MESH else t,
                         records_2015_2025=n, development_found=len(hits),
                         beyond_data_sharing=len(hits - found['"data sharing"'][1]),
                         only_this_term=len(hits - others), limit=limit(db),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing.csv", rows)

#### Runs 2 and 3. Small blocks of factor terms

**Question.** Which few factor terms, paired with the phrase, find the most development sources for the fewest records?

**Measured.** Run 2 pairs each candidate factor term with the phrase, anywhere in the record and, in Scopus and Web of Science, within 5 words of it, then builds a block one term at a time, each time adding the term with the most new sources per extra record. Run 3 tries every combination of up to seven terms offline, from the sources each term finds.

**Result.** The best blocks have five to seven terms. Anywhere in the record, they find 18 sources in PubMed for 3,452 records, 24 in Scopus for 6,805 and 26 in Web of Science for 8,656.

**Decision.** A block that small, chosen by yield, looks picked: a reader asks why those terms and no others. Runs 5 to 9 test the whole list of candidate terms in its place, with six general words for a factor that Claude proposed: *factor\**, *influenc\**, *determinant\**, *driver\**, *concern\** and *trust*.

Files: [term_table_factors.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_factors.csv), [term_table_factor_combinations.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_factor_combinations.csv).

In [ ]:
#| eval: false
def sharing_and(db, factors, near=False):
    """The sharing block AND any factor term: anywhere in the record, or within 5 words."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if db == "PubMed":
        return field(db, "anywhere", SHARING[db]) + (f" AND {field(db, 'anywhere', factors)}" if factors else "")
    if not factors:
        return field(db, "anywhere", SHARING[db])
    if near:
        return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"
    return f"{field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)}"


# Run 2
rows = []
for db in DATABASES:
    dev = dev_in(db)
    base = sl.retrieved(db, sharing_and(db, []), dev)
    for near in ([False] if db == "PubMed" else [False, True]):  # PubMed has no proximity for truncated terms
        mode = "within 5 words" if near else "anywhere in title or abstract"
        found = {f: (sl.count(db, f"({sharing_and(db, [f], near)}) AND {limit(db)}"),
                     sl.retrieved(db, sharing_and(db, [f], near), dev)) for f in ALL_FACTORS}
        for f in ALL_FACTORS:
            rows.append(dict(database=db, placement=mode, step="", term=f, records_2015_2025=found[f][0],
                             development_found=len(found[f][1]), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in found[f][1]))))
        # the block one term at a time: most new studies per extra record first
        chosen, got, load = [], set(), 0
        while True:
            best = None
            for f in ALL_FACTORS:
                new = found[f][1] - got
                if f in chosen or not new:
                    continue
                n = sl.count(db, f"({sharing_and(db, chosen + [f], near)}) AND {limit(db)}")
                if best is None or len(new) / max(n - load, 1) > best[0]:
                    best = (len(new) / max(n - load, 1), f, n, new)
            if best is None:
                break
            _, f, load, new = best
            chosen.append(f)
            got |= new
            rows.append(dict(database=db, placement=mode, step=len(chosen), term=f, records_2015_2025=load,
                             development_found=len(got), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in got))))
sl.write("term_table_factors.csv", rows)

# Run 3: a combination's studies are the union of its terms' studies, so they are worked out offline; only the
# record counts of the best combinations go to the databases.
MAX_TERMS, VERIFY = 7, 3
table = [r for r in sl.read("term_table_factors.csv") if r["step"] == ""]
rows = []
for (db, mode), group in itertools.groupby(table, key=lambda r: (r["database"], r["placement"])):
    group = [r for r in group if int(r["development_found"]) > 0]
    near = mode == "within 5 words"
    studies = sorted({s for r in group for s in r["studies"].split("; ")})
    bit = {s: 1 << i for i, s in enumerate(studies)}
    terms = [(r["term"], int(r["records_2015_2025"]), sum(bit[s] for s in r["studies"].split("; ")))
             for r in group]
    best = {}
    for k in range(1, min(MAX_TERMS, len(terms)) + 1):
        for combo in itertools.combinations(terms, k):
            mask = 0
            for _, _, m in combo:
                mask |= m
            # records estimated as the sum of the terms' own records, to rank; the real count comes next
            best.setdefault(bin(mask).count("1"), []).append((sum(c[1] for c in combo), [c[0] for c in combo]))
    for n in sorted(best, reverse=True):
        for _, combo in sorted(best[n])[:VERIFY]:
            rows.append(dict(database=db, placement=mode, studies_kept=n,
                             records_2015_2025=sl.count(db, f"({sharing_and(db, combo, near)}) AND {limit(db)}"),
                             terms=" OR ".join(combo)))
sl.write("term_table_factor_combinations.csv", rows)

#### Run 4. The phrase in the title, or anywhere

**Question.** Where must the phrase appear? Every factor term combines with it, so this comes before any choice of factor terms.

**Measured.** The phrase in the title alone, and anywhere in title, abstract and keywords: records and development sources found.

**Result.** The title alone cuts records by about 80%, and loses 4 development sources in PubMed, 12 in Scopus and 13 in Web of Science.

**Decision.** The phrase alone in the title loses too many sources. Factor terms are tested with the phrase anywhere in the record (run 5).

Files: [term_table_sharing_field.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing_field.csv).

In [ ]:
#| eval: false
rows = []
for db in DATABASES:
    for where in ("title", "anywhere"):
        q = field(db, where, SHARING[db])
        hits = sl.retrieved(db, q, dev_in(db))
        rows.append(dict(database=db, field=where, query=q, limit=limit(db),
                         records_2015_2025=sl.count(db, f"{q} AND {limit(db)}"), development_found=len(hits),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_field.csv", rows)

#### Run 5. Phrase plus factor words

**Question.** Does adding factor terms to the phrase find more development sources, and at what cost?

**Measured.** Three versions, each with the whole block of candidate factor terms:

1. the phrase in the title, with no factor terms;
2. the phrase anywhere AND a factor term anywhere;
3. version 1 OR version 2, which later runs call phrase plus factor words.

Versions 2 and 3 each run once with the full block and once without the five terms that find no development source in any database: *facilitator\**, *driver\**, *interoperab\**, *"open research"* and *"digital curation"*.

**Result.** Records for 2015 to 2025, and development sources found, without those five terms:

| Version | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| 1. The phrase, title | 1,677 and 17 | 5,517 and 16 | 4,438 and 16 |
| 2. The phrase and a factor term, anywhere | 6,959 and 18 | 22,134 and 24 | 15,057 and 26 |
| 3. Version 1 OR version 2 | 7,525 and 21 | 23,357 and 25 | 16,208 and 27 |

The five terms add records and no sources in every database.

**Decision.** The five terms go. All three versions survive the rule, each finding more sources than the one before at more records.

Files: [term_table_structure.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_structure.csv).

In [ ]:
#| eval: false
blocks = {"full": ALL_FACTORS, "full minus zero-study terms": [t for t in ALL_FACTORS if t not in ZERO_STUDY]}
rows = []
for db in DATABASES:
    dev = dev_in(db)
    res = {"1 title only": field(db, "title", SHARING[db])}
    for b, factors in blocks.items():
        res[f"2 anywhere AND factor, {b}"] = (f"({field(db, 'anywhere', SHARING[db])} AND "
                                              f"{field(db, 'anywhere', factors)})")
        res[f"3 version 1 OR version 2, {b}"] = version3(db, factors, near=False)
    res = {k: (q, sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)) for k, q in res.items()}
    for k, (q, n, hits) in res.items():
        beaten = [o for o, (_, n2, h2) in res.items() if o != k and len(h2) >= len(hits) and n2 < n]
        rows.append(dict(database=db, version=k, query=q, limit=limit(db), records_2015_2025=n,
                         development_found=len(hits), of=len(dev), beaten_by="; ".join(beaten),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_structure.csv", rows)

#### Run 6. What each factor term costs

**Question.** Does any single factor term carry the load, or cost much on its own?

**Measured.** Phrase plus factor words with the 29 remaining terms, removing one term at a time and putting it back: the records each term alone adds, and the development sources no other term finds.

**Result.** Only *repositor\** in PubMed (Coady and Wagner 2013) and *challenge\** in Web of Science (Van Horn and Gazzaniga 2013) find a source no other term finds. Removing *privacy* loses no source in any database.

**Decision.** *privacy* leaves the block. The screening guide keeps privacy as a factor that obstructs sharing.

Files: [term_table_leave_one_out.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_leave_one_out.csv).

In [ ]:
#| eval: false
factors = [t for t in ALL_FACTORS if t not in ZERO_STUDY]
rows = []
for db in DATABASES:
    dev = dev_in(db)
    q = version3(db, factors, near=False)
    n0, h0 = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
    for t in factors:
        q = version3(db, [f for f in factors if f != t], near=False)
        n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
        lost = sorted(LABEL[r] for r in h0 - h)
        rows.append(dict(database=db, term_removed=t, records_with_all=n0, records_saved=n0 - n,
                         studies_with_all=len(h0), studies_lost=len(lost), lost="; ".join(lost)))
sl.write("term_table_leave_one_out.csv", rows)

#### Run 7. Removing factor terms one after another, and proximity

**Question.** Can removing factor terms bring the load down without losing sources, and does requiring the factor term near the phrase help?

**Measured.** Phrase plus factor words with 28 terms, removing one term per step without putting it back, in the order of run 6: terms that lose no source first, most records saved first. In Scopus and Web of Science the run repeats with the factor term within 5 words of the phrase.

**Result.** Sources drop out by the second or third step in Scopus and Web of Science. With one factor term left, the query still returns 2,209 records in PubMed, 5,632 in Scopus and 9,051 in Web of Science. The order removes *polic\** and *barrier\** early, because they cost the most records. Proximity halves the load: with all 28 terms, 9,018 records in Scopus against 21,664, and 7,175 in Web of Science against 15,294.

**Decision.** Cutting terms cannot bring the load down far, and a query about factors without *polic\** and *barrier\** makes little sense to a reader. Phrase plus factor words keeps all 28 terms, within 5 words of the phrase in Scopus and Web of Science.

Files: [term_table_cuts.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_cuts.csv).

In [ ]:
#| eval: false
loo = sl.read("term_table_leave_one_out.csv")
rows = []
for db in DATABASES:
    dev = dev_in(db)
    order = [r["term_removed"] for r in sorted((r for r in loo if r["database"] == db),
             key=lambda r: (int(r["studies_lost"]), -int(r["records_saved"])))
             if r["term_removed"] not in OUT_OF_SCOPE]
    for near in ([False] if db == "PubMed" else [False, True]):
        kept = list(order)
        h_start = sl.retrieved(db, version3(db, kept, near), dev)
        for step in range(len(order)):
            cut = order[step - 1] if step else ""
            if cut:
                kept.remove(cut)
            q = version3(db, kept, near)
            n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
            rows.append(dict(database=db, placement="within 5 words" if near else "anywhere", step=step,
                             term_cut=cut, terms_left=len(kept), records_2015_2025=n, development_found=len(h),
                             of=len(dev), lost_so_far="; ".join(sorted(LABEL[r] for r in h_start - h)),
                             query=q, limit=limit(db)))
sl.write("term_table_cuts.csv", rows)

#### Run 8. Records of phrase plus factor words

**Question.** How many records does phrase plus factor words return across the three databases?

**Measured.** Phrase plus factor words with 28 terms, anywhere in PubMed and within 5 words in Scopus and Web of Science: every record the three return, counted once duplicates are removed. Duplicates here share a DOI, or a title and year; bib-dedupe, which deduplicates the review's records, finds as many or more.

**Result.** 23,488 records, or 14,255 once duplicates are removed.

**Decision.** Too many to screen. Run 11 sets it against the queries that replace it.

Files: [final_query_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/final_query_records.csv).

In [ ]:
#| eval: false
for db in ["Scopus", "Web of Science"]:
    lost = sl.retrieved(db, version3(db, FACTORS, False), dev_in(db)) - sl.retrieved(db, version3(db, FACTORS, True), dev_in(db))
    print(f"{db}, found anywhere but not within 5 words: {'; '.join(sorted(LABEL[r] for r in lost)) or '-'}")

recs = []
for db in DATABASES:
    q = f"{version3(db, FACTORS, near=db != 'PubMed')} AND {limit(db)}"
    if db == "PubMed":
        got = list(sl.pubmed_summary(sl.pubmed_all_ids(q)).values())
    elif db == "Scopus":
        got = sl.scopus_records(q)
    else:
        got = sl.wos_records(q)
    recs += [dict(r, database=db) for r in got]
sl.write("final_query_records.csv", recs)

#### Run 9. Development sources phrase plus factor words misses

**Question.** Which development sources does phrase plus factor words miss, and why?

**Measured.** Each development source with a DOI or PMID, checked in each database. For each source none finds, whether the databases index it.

**Result.** The three databases together find 28 of 40. Five are in none of the databases. Seven are indexed and missed: Borgman (2012), Cohn (2012), Costello (2009), Haddow et al. (2011), McCullough (2009), Parr (2007) and Zimmerman (2008). Three of them word sharing in ways the phrase does not match.

**Decision.** Run 10 tests a form of *share* near *data* in place of the phrase.

Files: [final_query_missed.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/final_query_missed.csv).

In [ ]:
#| eval: false
found, indexed = {}, {}
for db in DATABASES:
    found[db] = sl.retrieved(db, version3(db, FACTORS, near=db != "PubMed"), dev_in(db))
    indexed[db] = {s["ref_id"] for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
anyf = set().union(*found.values())

rows = []
for s in sorted(DEV, key=lambda s: s["label"]):
    if s["ref_id"] in anyf:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    rows.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                     pmid=s["pmid"], field=s["field"], source_type=s["source_type"],
                     pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("final_query_missed.csv", rows)

### Share near data

#### Run 10. Share near data, in place of the phrase

**Question.** Run 9 found three development sources missed because their titles word sharing differently: "sharing research data" (Borgman 2012), "sharing of health data" (Haddow et al. 2011) and "sharing and reuse of ecological data" (Zimmerman 2008). Does share near data find them, and at what cost?

**Measured.** The phrase against share within 3 and within 5 words of *data*, in the title and anywhere in the record. PubMed's proximity search takes no truncation, so there the block writes out *sharing*, *shared*, *share* and *shares*.

**Result.** Records for 2015 to 2025, and development sources found:

| | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| The phrase, title | 1,677 and 17 | 5,517 and 16 | 4,438 and 16 |
| Share within 3 words, title | 2,385 and 20 | 8,725 and 26 | 7,148 and 27 |
| Share within 5 words, title | 2,532 and 20 | 9,547 and 27 | 7,813 and 28 |
| Share within 5 words, anywhere | 18,840 and 21 | 82,908 and 30 | 62,198 and 31 |

In the title, share near data finds 26 to 28 development sources in Scopus and Web of Science with no factor words, where phrase plus factor words finds 21 for about the same records. Anywhere, it returns 7 to 9 times the records for one to three more sources. Web of Science's API rejects the language limit beside a proximity search, so its counts here cover all languages; elsewhere the limit removes about 0.5% of records.

**Decision.** Share near data in the title replaces the phrase and the factor words. Searching anywhere stays out. Run 11 compares the 3- and 5-word versions across the three databases together.

Files: [term_table_sharing_proximity.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_sharing_proximity.csv).

In [ ]:
#| eval: false
rows = []
for db in DATABASES:
    dev = dev_in(db)
    phrase = {}
    for where in ("title", "anywhere"):
        for n in (0, 3, 5):
            q = field(db, where, SHARING[db]) if n == 0 else share_near_data(db, where, n)
            lim = limit(db) if n == 0 else limit_near(db)
            recs, hits = sl.count(db, f"({q}) AND {lim}"), sl.retrieved(db, q, dev)
            if n == 0:
                phrase[where] = hits
            rows.append(dict(database=db, field=where, sharing="phrase" if n == 0 else f"within {n} words",
                             query=q, limit=lim, records_2015_2025=recs, development_found=len(hits),
                             beyond_phrase="; ".join(sorted(LABEL[r] for r in hits - phrase[where])),
                             studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_proximity.csv", rows)

#### Run 11. Candidate queries across the three databases

**Question.** Across the three databases together, which query finds the most development sources for the fewest records?

**Measured.** Records with duplicates removed, and development sources at least one database finds, out of the 35 at least one indexes, for phrase plus factor words and three candidates:

1. share within 3 words of *data*, title;
2. share within 5 words of *data*, title;
3. candidate 2 OR the factor part of phrase plus factor words.

**Result.**

| | Records | Without duplicates | Development sources found |
|:--|--:|--:|--:|
| Phrase plus factor words | 23,488 | 14,255 | 28 |
| 1. Share within 3 words, title | 18,258 | 9,591 | 28 |
| 2. Share within 5 words, title | 19,892 | 10,493 | 29 |
| 3. Candidate 2 OR factor part | 30,985 | 18,135 | 31 |

Candidate 2 adds Zimmerman (2008) for 902 records; candidate 3 adds Fisher and Fortmann (2010) and Piwowar (2011) for 7,642 more. Cohn (2012), Costello (2009), McCullough (2009) and Parr (2007) are missed by every query: their titles speak of open sourcing, online publication, open access journals and DataONE.

**Decision.** Phrase plus factor words drops out: candidate 1 finds as many sources in about 4,700 fewer records. Candidate 2 is chosen; the two sources candidate 3 adds do not justify 7,642 more records, so factor terms guide screening.

Files: [candidate_queries.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/candidate_queries.csv).

In [ ]:
#| eval: false
in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}


def factor_part(db):
    """The factor part of version 3: anywhere in PubMed, within 5 words in Scopus and Web of Science."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(FACTORS)
    if db == "PubMed":
        return f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', FACTORS)})"
    return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"


CANDIDATES = {"1 title, within 3 words": lambda db: share_near_data(db, "title", 3),
              "2 title, within 5 words": lambda db: share_near_data(db, "title", 5),
              "3 version 2 OR factor part": lambda db: f"({share_near_data(db, 'title', 5)} OR {factor_part(db)})"}
summary, all_recs = [], []
for name, make in CANDIDATES.items():
    found, recs = set(), []
    for db in DATABASES:
        q = make(db)
        hits = sl.retrieved(db, q, dev_in(db))
        got = fetch(db, q, limit_near(db))
        recs += [dict(r, database=db, candidate=name) for r in got]
        found |= hits
        summary.append(dict(candidate=name, database=db, query=q, limit=limit_near(db), records_2015_2025=len(got),
                            development_found=len(hits), studies="; ".join(sorted(LABEL[r] for r in hits))))
    summary.append(dict(candidate=name, database="all three", records_2015_2025=len(recs),
                        unique_records=unique_count(recs), development_found=len(found), of_indexed=len(in_any),
                        missed_indexed="; ".join(sorted(LABEL[r] for r in in_any - found))))
    all_recs += recs
sl.write("candidate_queries.csv", summary)
sl.write("candidate_records.csv", all_recs)

### Narrowing

#### Run 12. Document types

**Question.** Could a document type limit narrow candidate 2 without dropping sources the review keeps?

**Measured.** Candidate 2's records by each database's document type, and the database's type for each development source it finds, beside the source type the coding gave it.

**Result.** Editorials, letters, notes and news make up about 18% of the records in PubMed, 8% in Scopus and 11% in Web of Science. The databases' types do not follow the coded source types: most opinion pieces are indexed as Article or Review, and a Note or Editorial can be a policy document (Harding et al. 2013) or an empirical study (Huang et al. 2013).

**Decision.** No document type limit. A limit would keep most opinion pieces and drop sources the review keeps, and the protocol makes all source types eligible.

Files: [document_types.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/document_types.csv).

In [ ]:
#| eval: false
import os
import collections

Q2 = {db: share_near_data(db, "title", 5) for db in DATABASES}
rows = []


def record(db, counts, total, known):
    for t, n in counts.items():
        rows.append(dict(database=db, what="records", document_type=t, n=n, of=total))
    for r, t in known.items():
        rows.append(dict(database=db, what="development study", document_type=t, label=LABEL[r],
                         source_type=next(s["source_type"] for s in DEV if s["ref_id"] == r)))


# PubMed: publication types overlap, so each count is the records carrying that type
db, q = "PubMed", f"({Q2['PubMed']}) AND {limit('PubMed')}"
pm_types = ["Journal Article", "Review", "Editorial", "Comment", "Letter", "News", "Published Erratum",
            "Practice Guideline", "Guideline", "Systematic Review", "Meta-Analysis"]
counts = {t: sl.count(db, f'{q} AND "{t}"[pt]') for t in pm_types}
counts["none of Editorial, Comment, Letter, News"] = sl.count(db, f"{q} NOT (Editorial[pt] OR Comment[pt] OR Letter[pt] OR News[pt])")
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    _, ids = sl.pubmed_search(sl.id_clause(db, [s for s in DEV if s["ref_id"] == r]), retmax=1)
    j = sl._get(sl.EU + "esummary.fcgi", dict(db="pubmed", id=ids[0], retmode="json",
                api_key=os.environ["NCBI_API_KEY"]), pause=0.11).json()["result"][ids[0]]
    known[r] = ", ".join(j.get("pubtype", []))
record(db, counts, sl.count(db, q), known)

# Scopus: one document type per record
db, q = "Scopus", f"({Q2['Scopus']}) AND {limit('Scopus')}"
sc_types = {"ar": "Article", "re": "Review", "ed": "Editorial", "le": "Letter", "no": "Note", "sh": "Short survey",
            "cp": "Conference paper", "ch": "Book chapter", "bk": "Book", "er": "Erratum", "cr": "Conference review",
            "dp": "Data paper"}
counts = {name: sl.count(db, f"{q} AND DOCTYPE({code})") for code, name in sc_types.items()}
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    j = sl._get(sl.SC, dict(query=f"DOI({doi})", field="doi,subtypeDescription"),
                {"X-ELS-APIKey": os.environ["SCOPUS_API_KEY"], "Accept": "application/json"}, pause=0.12).json()
    known[r] = j["search-results"]["entry"][0].get("subtypeDescription", "?")
record(db, counts, sl.count(db, q), known)

# Web of Science: types read from every record; a record can carry more than one
db, q = "Web of Science", f"({Q2['Web of Science']}) AND {limit_near('Web of Science')}"
counts, total, page = collections.Counter(), 0, 1
while True:
    j = sl.wos_search(q, limit=50, page=page)
    for h in j.get("hits", []):
        counts.update(h.get("types", ["?"]))
        total += 1
    if page * 50 >= j["metadata"]["total"] or not j.get("hits"):
        break
    page += 1
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    h = sl.wos_search(f"DO=({doi})", limit=1)["hits"]
    known[r] = ", ".join(h[0].get("types", ["?"])) if h else "?"
record(db, dict(counts), total, known)
sl.write("document_types.csv", rows)

#### Run 13. Data sharing in the computing sense

**Question.** How many of candidate 2's records concern computing, such as encryption schemes, blockchain frameworks or cloud storage, and do technical words pick them out?

**Measured.** Candidate 2's records whose title carries one of 16 technical words, the development sources among them, and 25 Scopus titles read to judge whether the words mark the computing sense.

**Result.** The technical words mark 6% of the records in PubMed, 35% in Scopus and 29% in Web of Science, and no development source. The sample titles are almost all technical. *scheme* also catches governance and funding schemes, *secure* catches secure access to research data, and *federated learning* marks an alternative to sharing health research data.

**Decision.** *scheme*, *secure* and *federated learning* leave the list. Run 14 tests a NOT clause on the 13 words that remain.

Files: [technical_sense.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/technical_sense.csv).

In [ ]:
#| eval: false
import os

TECH_WORDS = ["encrypt*", "cryptograph*", "blockchain*", "cloud", '"access control"', "attribute-based",
              '"privacy-preserving"', '"privacy preserving"', '"smart contract*"', '"internet of things"', "IoT",
              "vehicular", '"edge computing"', '"federated learning"', "secure", "scheme"]
rows = []
for db in DATABASES:
    q, lim = f"({share_near_data(db, 'title', 5)})", limit_near(db)
    total = sl.count(db, f"{q} AND {lim}")
    any_tech = field(db, "title", TECH_WORDS)
    lost = sl.retrieved(db, f"{q} AND {any_tech}", dev_in(db))
    rows.append(dict(database=db, term="any", records=sl.count(db, f"{q} AND {any_tech} AND {lim}"), of=total,
                     development_studies="; ".join(sorted(LABEL[r] for r in lost))))
    for t in TECH_WORDS:
        rows.append(dict(database=db, term=t, records=sl.count(db, f"{q} AND {field(db, 'title', [t])} AND {lim}"),
                         of=total))
    if db == "Scopus":  # titles to judge whether the words pick out the computing sense
        j = sl._get(sl.SC, dict(query=f"{q} AND {any_tech} AND {lim}", count=25, field="dc:title"),
                    {"X-ELS-APIKey": os.environ["SCOPUS_API_KEY"], "Accept": "application/json"}).json()
        for e in j["search-results"].get("entry", []):
            print(e.get("dc:title", ""))
sl.write("technical_sense.csv", rows)

#### Run 14. Leaving out computing words

**Question.** What does a NOT clause on the 13 remaining technical words remove?

**Measured.** Candidate 2 with a NOT clause on *blockchain\**, *cloud*, *encrypt\**, *"privacy-preserving"*, *"privacy preserving"*, *IoT*, *"access control"*, *attribute-based*, *"internet of things"*, *vehicular*, *cryptograph\**, *"edge computing"* and *"smart contract\*"*: the records kept and removed, the development sources kept, and every removed title that mentions research, health or science.

**Result.** The clause cuts the records from 10,493 to 7,505 and keeps all 29 development sources. But 698 removed titles mention research, health or science, and some concern the factors the review maps, such as "Role of Incentives in the Use of Blockchain-Based Platforms for Sharing Sensitive Health Data". Most of this literature came after 2014, so the development set cannot show what the clause costs.

**Decision.** A clause this broad removes sources the review wants. Run 15 builds a narrower clause tied to an eligibility criterion.

Files: [technical_filter_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/technical_filter_records.csv), [technical_filter_flagged.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/technical_filter_flagged.csv).

In [ ]:
#| eval: false
TECH = ["blockchain*", "cloud", "encrypt*", '"privacy-preserving"', '"privacy preserving"', "IoT",
        '"access control"', "attribute-based", '"internet of things"', "vehicular", "cryptograph*",
        '"edge computing"', '"smart contract*"']
RESEARCH = re.compile(r"research|clinical|health|medical|biomedical|genom|patient|scien|trial|cohort|"
                      r"hospital|epidemiolog|biobank|neuro", re.I)


def without_tech(db):
    q, t = share_near_data(db, "title", 5), field(db, "title", TECH)
    return f"(({q}) AND NOT {t})" if db == "Scopus" else f"(({q}) NOT {t})"


in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
kept_recs, removed_recs, found = [], [], set()
for db in DATABASES:
    found |= sl.retrieved(db, without_tech(db), dev_in(db))
    kept_recs += [dict(r, database=db, status="kept") for r in fetch(db, without_tech(db), limit_near(db))]
    removed_recs += [dict(r, database=db, status="removed") for r in
                     fetch(db, f"(({share_near_data(db, 'title', 5)}) AND {field(db, 'title', TECH)})", limit_near(db))]
print(f"{len(kept_recs)} kept, {unique_count(kept_recs)} without duplicates; "
      f"{len(found)} of {len(in_any)} indexed development studies found")

seen, flagged = set(), []
for r in removed_recs:  # removed titles that look like research data, one per title
    key = re.sub(r"[^a-z0-9]", "", r["title"].lower())
    if r["title"] and RESEARCH.search(r["title"]) and key not in seen:
        seen.add(key)
        flagged.append(r)
sl.write("technical_filter_records.csv", kept_recs + removed_recs)
sl.write("technical_filter_flagged.csv", flagged)

#### Run 15. A NOT clause on systems, products and services

**Question.** The eligibility criteria gain a line: sources that design, build or evaluate a specific system, product or service for sharing data are out of scope. Which title words almost always mark such a source, so the clause removes only what screening would exclude?

**Measured.** The words added in four groups, each on top of the last: records with duplicates removed, records each group saves, and development sources found.

**Result.**

| Words left out of titles | Records | Saved | Development sources found |
|:--|--:|--:|--:|
| None | 10,493 | | 29 |
| *blockchain\**, *"smart contract\*"* | 9,157 | 1,336 | 29 |
| + *IoT*, *"internet of things"*, *vehicular*, *"edge computing"* | 8,848 | 309 | 29 |
| + *cloud* | 7,933 | 915 | 29 |
| + *attribute-based*, *cryptograph\** | 7,857 | 76 | 29 |

**Decision.** The clause stops at *cloud*. *attribute-based* and *cryptograph\** save 76 records, too few for two more words a reader must parse, and screening excludes such records anyway. *encrypt\**, *"access control"* and *"privacy-preserving"* also mark governance of research data, so they stay out of the clause. A study of why people adopt such a system concerns factors and is in scope; the clause removes it, a known cost.

Files: [query_exclusion.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/query_exclusion.csv).

In [ ]:
#| eval: false
BLOCKCHAIN = ["blockchain*", '"smart contract*"']
DEVICES = ["IoT", '"internet of things"', "vehicular", '"edge computing"']
STEPS = [("blockchain", BLOCKCHAIN), ("+ devices", BLOCKCHAIN + DEVICES),
         ("+ cloud", BLOCKCHAIN + DEVICES + ["cloud"]),
         ("+ attribute-based, cryptograph*", BLOCKCHAIN + DEVICES + ["cloud", "attribute-based", "cryptograph*"])]


def share_not(db, terms):
    """A form of share within 5 words of data in the title, NOT any of the terms in the title."""
    q, t = share_near_data(db, "title", 5), field(db, "title", terms)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
rows, summary = [], []
for name, terms in STEPS:
    recs, found = [], set()
    for db in DATABASES:
        q = share_not(db, terms)
        found |= sl.retrieved(db, q, dev_in(db))
        got = fetch(db, q, limit_near(db))
        recs += [dict(r, database=db, version=name) for r in got]
        summary.append(dict(version=name, database=db, query=q, limit=limit_near(db), records_2015_2025=len(got)))
    summary.append(dict(version=name, database="all three", records_2015_2025=len(recs), unique_records=unique_count(recs),
                        development_found=len(found), of_indexed=len(in_any),
                        missed_indexed="; ".join(sorted(LABEL[r] for r in in_any - found))))
    rows += recs
sl.write("query_exclusion.csv", summary)
sl.write("query_exclusion_records.csv", rows)

### The query

```
Title: share* NEAR/5 data
NOT
Title: blockchain* OR "smart contract*" OR IoT OR "internet of things" OR vehicular OR "edge computing" OR cloud

Limits: published 2015 to 2025; English or French
```

**PubMed**

```text
("sharing data"[ti:~5] OR "shared data"[ti:~5] OR "share data"[ti:~5] OR "shares data"[ti:~5]) NOT (blockchain*[ti] OR "smart contract*"[ti] OR IoT[ti] OR "internet of things"[ti] OR vehicular[ti] OR "edge computing"[ti] OR cloud[ti])
```

**Scopus**

```text
TITLE(shar* W/5 data) AND NOT (TITLE(blockchain*) OR TITLE("smart contract*") OR TITLE(IoT) OR TITLE("internet of things") OR TITLE(vehicular) OR TITLE("edge computing") OR TITLE(cloud))
```

**Web of Science**

```text
TI=(shar* NEAR/5 data) NOT (TI=(blockchain*) OR TI=("smart contract*") OR TI=(IoT) OR TI=("internet of things") OR TI=(vehicular) OR TI=("edge computing") OR TI=(cloud))
```

PubMed's proximity search takes no truncation, so its block writes out four forms of *share*. Reuse, the other concept part of the PCC table, is left out: the review asks about sharing.

## Testing the query

Run 16 measures the query's recall on the test set, held out until the query was chosen. Run 17 is kept as a record.

### Run 16. The test set

**Question.** How much of the held-out test set does the query find?

**Measured.** The query against the 31 test sources, checked by DOI or PMID in each database, with a Wilson 95% interval. No earlier run used the test set. Its misses had been seen on 16 November, during the first attempt, before this query existed; the query was not built from them.

**Result.**

| | Found | 95% interval |
|:--|--:|:--|
| PubMed | 8 of 11 | 0.43 to 0.90 |
| Scopus | 16 of 23 | 0.49 to 0.84 |
| Web of Science | 12 of 18 | 0.44 to 0.84 |
| At least one database | 18 of 25 | 0.52 to 0.86 |

The query misses 13 test sources. Six cannot be found by any query: Bishoff and Johnston (2015), He (2016) and Zvyagintseva (2015) are in none of the databases, and Brown et al. (2015), Guédon (2015) and Koopman (2015) have no DOI or PMID. The other seven name research data management, open data or open science in their titles. These figures count every eligible test source, which follows from opening tier 2 to all source types (runs 19 and 20); with tier 2 limited, 10 test sources were in scope and the query found 9.

**Decision.** All 13 are added by hand. Run 21 measures title terms that would catch the seven.

Files: [test_coverage.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage.csv), [test_missed.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_missed.csv).

In [ ]:
#| eval: false
QUERY_NOT = BLOCKCHAIN + DEVICES + ["cloud"]
test = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in test if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, share_not(db, QUERY_NOT), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=share_not(db, QUERY_NOT), found=len(found[db]), indexed=len(indexed[db]),
                     coverage=p, low=lo, high=hi))
any_found, in_any = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(any_found), len(in_any))
rows.append(dict(database="all three", found=len(any_found), indexed=len(in_any), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(test, key=lambda s: s["label"]):
    if s["ref_id"] in any_found:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], in_scope=s["in_scope"],
                       pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage.csv", rows)
sl.write("test_missed.csv", missed)

### Run 17. Known items

*The known-item list was generated by a language model and not independently verified, so it does not validate the query. This run is kept as a record.*

**Question.** Does the query find the sources on a known-item list of 11 health research sources, 5 cross-disciplinary reviews, 3 adjacent sources and 2 comparators?

**Measured.** Each item checked by DOI in each database.

**Result.** The query finds 8 of the 11 health research sources, all 5 reviews, and all adjacent and comparator sources. It misses Hutchings et al. (2020), which puts seven words between *sharing* and *data*, and Swales et al. (2024) and Inau et al. (2023), on a data transfer agreement and health data stewardship.

**Decision.** The three misses are eligible and go into the records by hand.

Files: [known_item_test.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/known_item_test.csv).

In [ ]:
#| eval: false
QUERY = {db: share_not(db, QUERY_NOT) for db in DATABASES}  # run 15's NOT clause up to cloud
ITEMS = [("tier 1", "Rowhani-Farid et al. 2017", "10.1186/s41073-017-0028-9"),
         ("tier 1", "Devriendt et al. 2021", "10.1089/bio.2020.0037"),
         ("tier 1", "Kalkman et al. 2019", "10.1186/s12910-019-0359-9"),
         ("tier 1", "Hamilton et al. 2023", "10.1136/bmj-2023-075767"),
         ("tier 1", "Hutchings et al. 2020", "10.1186/s13643-020-01485-5"),
         ("tier 1", "Saulnier et al. 2019", "10.1038/s41597-019-0310-4"),
         ("tier 1", "Gudi et al. 2022", "10.2196/33591"),
         ("tier 1", "Swales et al. 2024", "10.3389/fphar.2024.1332700"),
         ("tier 1", "Andanda and Mlotshwa 2024", "10.1177/17470161241261907"),
         ("tier 1", "Inau et al. 2023", "10.2196/45013"),
         ("tier 1", "Scheibner et al. 2023", "10.1007/s10916-023-02009-z"),
         ("tier 2", "Fecher et al. 2015", "10.1371/journal.pone.0118053"),
         ("tier 2", "Zuiderwijk et al. 2020", "10.1371/journal.pone.0239283"),
         ("tier 2", "Perrier et al. 2020", "10.1371/journal.pone.0229182"),
         ("tier 2", "Chawinga and Zinn 2019", "10.1016/j.lisr.2019.04.004"),
         ("tier 2", "Thøgersen and Borlund 2022", "10.1108/JD-01-2021-0015"),
         ("adjacent", "Alam et al. 2024", "10.1371/journal.pdig.0000652"),
         ("adjacent", "Baines et al. 2024", "10.2196/50421"),
         ("adjacent", "Cascini et al. 2024", "10.1016/j.eclinm.2024.102551"),
         ("comparator", "van Panhuis et al. 2014", "10.1186/1471-2458-14-1144"),
         ("comparator", "Yehudi et al. 2025", "10.1017/dap.2024.79")]
items = [dict(ref_id=f"k{i}", group=g, label=l, doi=d.lower(), pmid="") for i, (g, l, d) in enumerate(ITEMS)]
for db in DATABASES:
    found = sl.retrieved(db, QUERY[db], items)
    for it in items:
        it[db] = ("found" if it["ref_id"] in found else
                  "missed" if sl.count(db, sl.id_clause(db, [it])) > 0 else "not indexed")
sl.write("known_item_test.csv", [dict(group=it["group"], label=it["label"], doi=it["doi"],
                                      found_any=any(it[db] == "found" for db in DATABASES),
                                      pubmed=it["PubMed"], scopus=it["Scopus"], web_of_science=it["Web of Science"])
                                 for it in items])

## Misses and scope

Runs 18 and 21 measure title terms that would catch sources the query misses. Runs 19 and 20 measure what opening tier 2 to all source types would cost in screening.

### Run 18. Governance terms

*The known-item list was generated by a language model and not independently verified, so it does not validate the query. This run is kept as a record.*

**Question.** Would title terms for the governance mechanisms the PCC concept names catch Swales et al. (2024) and Inau et al. (2023), and at what cost?

**Measured.** Five title terms, each with the NOT clause, against the query's 7,933 records: the records each adds once duplicates are removed, and the known sources it finds that the query misses.

**Result.**

| Title term | Records added | Known sources added |
|:--|--:|:--|
| *"data access"* | 1,077 | none |
| *"data transfer"* | 1,617 | Swales et al. (2024) |
| *"data governance"* | 1,120 | none |
| *"data stewardship"* | 82 | Inau et al. (2023) |
| *"FAIR data"*, *"FAIR principles"*, *"FAIR guiding principles"* | 425 | none |
| All five | 4,304 | |

**Decision.** None enters the query. A term added to catch one source looks picked, and the query stays one idea: a title that names sharing data.

Files: [governance_terms.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/governance_terms.csv).

In [ ]:
#| eval: false
TERMS = {'"data access"': ['"data access"'], '"data transfer"': ['"data transfer"'],
         '"data governance"': ['"data governance"'], '"data stewardship"': ['"data stewardship"'],
         "FAIR": ['"FAIR data"', '"FAIR principles"', '"FAIR guiding principles"']}


def not_tech(db, q):
    t = field(db, "title", QUERY_NOT)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


known = [dict(ref_id=f"k{i}", label=r["label"], doi=r["doi"], pmid="")
         for i, r in enumerate(sl.read("known_item_test.csv")) if r["group"] in ("tier 1", "tier 2")]
label = LABEL | {k["ref_id"]: k["label"] for k in known}
base_recs = [r for r in sl.read("query_exclusion_records.csv") if r["version"] == "+ cloud"]
base_u = unique_count(base_recs)
base_found = set().union(*(sl.retrieved(db, QUERY[db], dev_in(db)) | sl.retrieved(db, QUERY[db], known)
                           for db in DATABASES))
rows, all_new = [], []
for name, terms in TERMS.items():
    recs, found = [], set()
    for db in DATABASES:
        q = not_tech(db, field(db, "title", terms))
        found |= sl.retrieved(db, q, dev_in(db)) | sl.retrieved(db, q, known)
        recs += [dict(r, database=db, term=name) for r in fetch(db, q, limit_near(db))]
    rows.append(dict(term=name, records_added=unique_count(base_recs + recs) - base_u,
                     known_added="; ".join(sorted(label[r] for r in found - base_found))))
    all_new += recs
rows.append(dict(term="all five", records_added=unique_count(base_recs + all_new) - base_u))
sl.write("governance_terms.csv", rows)
sl.write("governance_terms_records.csv", all_new)

### Run 19. Tier 2: the records that could be included

**Question.** The protocol limited tier 2, research in other fields and cross-disciplinary research, to reviews and syntheses. How many records does that limit keep out?

**Measured.** Three counts that do not judge relevance: reference set sources that meet the criteria with and without the limit, the share of the query's records in PubMed, and Scopus's split of its records by subject area. Scopus applies AND NOT after every AND, so its limits come before the NOT clause.

**Result.** In the reference set, 30 sources meet every criterion apart from year with the limit and 76 without. Of the query's 7,933 records, 30% are in PubMed. Scopus splits its 7,067 records into 2,196 in health (31%), 84 reviews in other fields (1%) and 4,787 other records in other fields (68%).

**Decision.** The limit keeps out most records in other fields. Run 20 estimates how many of them would be included.

In [ ]:
#| eval: false
BASE = "TITLE(shar* W/5 data) AND PUBYEAR > 2014 AND PUBYEAR < 2026 AND LANGUAGE(english OR french)"
NOT = 'TITLE(blockchain* OR "smart contract*" OR IoT OR "internet of things" OR vehicular OR "edge computing" OR cloud)'
HEALTH = "SUBJAREA(MEDI OR NURS OR HEAL OR DENT OR PHAR OR IMMU OR NEUR OR BIOC)"  # Scopus's health subject areas


def n(extra=""):
    return sl.count("Scopus", f"(({BASE}) {extra}) AND NOT {NOT}")


total, health = n(), n(f"AND {HEALTH}")
other_reviews = n("AND DOCTYPE(re)") - n(f"AND DOCTYPE(re) AND {HEALTH}")
print(total, health, other_reviews, total - health - other_reviews)

### Run 20. Tier 2: the records that would be included

**Question.** How many more records would full-text screening take if tier 2 admitted all source types?

**Measured.** The local model judged 300 of the query's 7,933 records, drawn at random with seed 20261118, with `scope_prompt.md`, which states the screening guide's criteria without the tier criterion and asks for the field and whether the record is a review. Abstracts came from PubMed, Scopus or OpenAlex; 39 records had none. An earlier pass with `coding_prompt.md`, which does not state this review's criteria, counted computing systems and company data as research data sharing and was not used.

**Result.**

| Records the model includes | Of 300 | Share (95% interval) | Projected to 7,933 |
|:--|--:|:--|--:|
| Tier 2 limited to reviews and syntheses | 48 | 16% (12% to 21%) | 1,270 |
| Tier 2 open to all source types | 64 | 21% (17% to 26%) | 1,690 |
| Only with tier 2 open | 16 | 5% (3% to 9%) | 420 |

31 records are undecidable, 28 of them judged on title alone; counted as included, opening tier 2 adds 35 records, 930 projected. Most of the 16 concern factors in research data sharing outside health, such as "Data sharing in PLOS ONE: An analysis of Data Availability Statements".

**Decision.** Tier 2 opens to all source types: about a third more full-text screening, for the comparison the third question asks for.

Files: [scope_sample_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/scope_sample_records.csv), [scope_screening_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/scope_screening_qwen3.5-122b.csv).

In [ ]:
#| eval: false
# On the analysis server, with the local model running (scripts/launch_rds_llm.sh) and the sample drawn:
#   python3 analysis/00_search-development/local-model/coding.py scope_codes_qwen3.5-122b.csv scope_sample_records.csv
#   python3 analysis/00_search-development/local-model/scope.py scope_screening_qwen3.5-122b.csv
# The sample: one record per work from the query's records, drawn at random with seed 20261118.
import random

key, works = {}, []
order = {"PubMed": 0, "Scopus": 1, "Web of Science": 2}
recs = [r for r in sl.read("query_exclusion_records.csv") if r["version"] == "+ cloud"]
for r in sorted(recs, key=lambda r: order[r["database"]]):
    ks = [k for k in (("d", r["doi"]) if r["doi"] else None,
                      ("t", re.sub(r"[^a-z0-9]", "", r["title"].lower()) + r["year"]) if r["title"] else None) if k]
    g = next((key[k] for k in ks if k in key), None)
    if g is None:
        g = len(works)
        works.append(r)
    for k in ks:
        key[k] = g
sample = random.Random(20261118).sample(works, 300)  # abstracts are then fetched from PubMed, Scopus and OpenAlex

### Run 21. Research data management, open data and open science

**Question.** Five of the seven indexed test sources the query misses concern research data management, and two concern open data or open science. Would title terms for these catch them, and at what cost?

**Measured.** Four title terms, each with the NOT clause, against the query's 7,933 records: the records each adds once duplicates are removed, and the test sources it finds. For "research data management", the local model also judged the records it adds with `scope_prompt.md`.

**Result.**

| Title term | Records added | Test sources added |
|:--|--:|:--|
| *"research data management"* | 489 | Chigwada et al. (2017), Chiware and Mathe (2016), Schöpfel et al. (2018), Wittenberg and Elings (2017) |
| *"data management"* | 5,351 | the same four and Specht et al. (2015) |
| *"open data"* | 4,043 | Schmidt et al. (2016) |
| *"open science"* | 2,151 | Watson (2015) |

The model judged 214 of the 489 records "research data management" adds eligible (44%), 46 undecidable and 229 excluded, 136 of them as system designs, against 21% of the query's own records. This run draws on the test set, which no longer tests these terms independently.

**Decision.** None enters the query, which searches research data sharing alone. The screening guide lists research data management as a mechanism for implementation, and the test sources are added by hand.

Files: [management_terms.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/management_terms.csv), [rdm_screening_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/rdm_screening_qwen3.5-122b.csv), [rdm_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/rdm_records.csv).

In [ ]:
#| eval: false
# The title terms, measured as in run 18 with the test set added to the known sources:
#   TERMS = {'"research data management"', '"data management"', '"open data"', '"open science"'}
# writes management_terms.csv and management_terms_records.csv. The records research data management adds go, with
# abstracts, to rdm_records.csv, and the local model judges them:
#   python3 analysis/00_search-development/local-model/scope.py rdm_screening_qwen3.5-122b.csv rdm_records.csv

### Sources added by hand

The query misses 16 known sources that meet the review's criteria, the 13 test sources run 16 lists and three sources identified during search development, Hutchings et al. (2020), Swales et al. (2024) and Inau et al. (2023). Each goes into the records by hand, as `other-hand-added_<date>.ris` in `data/raw/search_exports/`. No term enters the query to catch them: runs 18 and 21 show that the terms that would catch them add 82 to 5,351 records each, and a term added for one source looks picked. The [search strategy](../../materials/search-strategy.qmd) page lists the 16.

## Terms the known sources use

### Run 22. Terms the sources use

**Question.** Which words do the known sources use for research data sharing, and does share near data cover them?

**Measured.** A local model, Qwen3.8-27B, read the title and abstract of each of the 102 sources in the reference set with `prompts/terms_prompt.md` and listed the words and phrases the source uses to describe research data sharing, copied exactly. The prompt names no factor and none of the screening guide's terms. Qwen3.5-122B did not fit on the free GPUs. The notebook drops terms that are not in the source's title or abstract, joins terms that differ only in case or hyphens, and counts the sources that use each term, in all 102 sources and in the development and test sets.

**Result.** The model returned 412 terms, 259 of them distinct; 29 appear in two or more sources, and 18 sources yield no term. The recurring terms fall into six patterns:

- **The same idea, worded differently:** "data sharing" (50 sources), "sharing data" (12), "share data" (11), "sharing of data" (4), "shared data" (2), "share their data" (2).
- **Sharing with the kind of data named:** "sharing research data" (4), "research data sharing" (3), "biodiversity data sharing" (3), "scientific data sharing" (2), "sharing individual level data" (2).
- **Sharing as a policy or practice:** "data sharing policies" (5), "data sharing practices" (3), "data sharing policy" (2), "data sharing program" (2), "data management and sharing practices" (2), "data sharers" (2).
- **Access without *share*:** "open data" (3), "data accessibility" (2), "making research data accessible" (2), "publish data" (2), "data repository" (2).
- **Management and curation:** "data curation" (3), "data management" (3), "data management plans" (3), "research data management services" (2).
- **Reuse:** "data reuse" (2).

The first three patterns hold *share* and *data* within a few words of each other. Each term in the last three appears in two or three sources.

Files: [terms_qwen3.8-27b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_qwen3.8-27b.csv), [terms_console_qwen3.8-27b.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_console_qwen3.8-27b.txt).

In [ ]:
# On the analysis server, with the local model running (scripts/launch_rds_llm.sh):
#   LLM_URL=http://<GPU server>:11451 python3 analysis/00_search-development/local-model/terms.py terms_qwen3.8-27b.csv
if path("terms_qwen3.8-27b.csv").exists():
    term_rows = read("terms_qwen3.8-27b.csv")
    sets = read("reference_sources.csv").set_index("ref_id")["set"]
    found = term_rows[term_rows["location"] != "not in text"].copy()
    found["term"] = found["term"].str.lower().str.replace(r"[-‐–]", " ", regex=True).str.split().str.join(" ")
    found = found.drop_duplicates(["ref_id", "term"])
    found["set"] = found["ref_id"].map(sets)
    table = (found.groupby("term")
             .agg(all=("ref_id", "size"),
                  development=("set", lambda s: int((s == "development").sum())),
                  test=("set", lambda s: int((s == "test").sum())))
             .sort_values(["all", "development"], ascending=False))
    print(f"{len(term_rows)} terms, {len(table)} distinct, {int((table['all'] > 1).sum())} used by two or more sources; "
          f"{int((term_rows['location'] == 'not in text').sum())} not in the text")
    none = sorted(set(sets.index) - set(term_rows["ref_id"]))
    print(f"{len(none)} sources with no term")
    display(table[table["all"] > 1])
else:
    print("Not run yet: terms_qwen3.8-27b.csv is missing.")

### Run 23. Table 1 terms in the sources

**Question.** How often do the terms in Table 1 of the screening guide appear in the titles and abstracts of the 102 sources, and how do they spread across the sources?

**Measured.** The notebook matches each Table 1 term against each source's title and abstract, with no model: a term ending in \* matches any word that starts with it, and a quoted term matches as a phrase. It counts the sources that use each term and each theme, in all 102 sources and in the development and test sets, and the number of themes each source touches.

**Result.** 79 sources use at least one Table 1 term and 23 use none; 12 of those 23 have a title only. The terms used most are *polic\** (28 sources), *challenge\** (18), *"research data management"* (14), *infrastructure* (12), and *barrier\**, *repositor\**, *"open access"* and *ethic\** (11 each). *facilitator\**, *interoperab\**, *"open research"* and *licens\** appear in no source. By theme, mechanisms for implementation appear in 42 sources, policies and what drive them in 39, factors that obstruct in 33, factors that facilitate in 29 and stakeholder perspectives in 14. Of the 83 sources with an abstract, 28 touch one theme and 44 touch two or more; of the 19 with a title only, 7 touch one theme and 12 none. *"research data management"* appears in 1 development source and 13 test sources.

Files: [reference_lookups.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_lookups.csv).

In [ ]:
TABLE1 = {
    "factors that obstruct": ["barrier*", "challenge*", "obstacle*", "reluctan*", "privacy"],
    "factors that facilitate": ["facilitator*", "enabler*", "incentive*", "motivat*", "infrastructure", "repositor*",
                                "interoperab*"],
    "stakeholder perspectives": ["perception*", "attitude*", "willingness"],
    "policies and what drive them": ["polic*", "mandate*", '"open science"', '"open data"', '"open access"',
                                     '"open research"'],
    "mechanisms for implementation": ["governance", "consent", "licens*", "ethic*", "stewardship",
                                      '"data management plan*"', '"data curation"', '"digital curation"',
                                      '"research data management"'],
}


def pattern(term):
    """A Table 1 term as a regular expression: * ends a word stem, quotes mark a phrase."""
    words = term.strip('"').split()
    parts = [re.escape(w[:-1]) + r"\w*" if w.endswith("*") else re.escape(w) for w in words]
    return re.compile(r"\b" + r"\s+".join(parts) + r"\b")


def plain(text):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text.lower())).strip()


if path("reference_lookups.csv").exists():
    lookups = read("reference_lookups.csv").set_index("ref_id")
    sources = read("reference_sources.csv").set_index("ref_id")
    text = (lookups["title"] + " " + lookups["abstract"]).map(plain)
    hits = pd.DataFrame({(theme, term): text.map(lambda t, p=pattern(term): bool(p.search(t)))
                         for theme, terms in TABLE1.items() for term in terms})
    hits.columns.names = ["theme", "term"]
    sets = sources["set"].reindex(hits.index)

    def counts(frame):
        return pd.DataFrame({"all": frame.sum(), "development": frame[sets == "development"].sum(),
                             "test": frame[sets == "test"].sum()}).astype(int)

    by_term = counts(hits)
    themes = hits.T.groupby(level="theme").any().T
    by_theme = counts(themes).reindex(TABLE1)
    per_source = themes.sum(axis=1)
    coded_on = lookups["abstract"].map(lambda a: "title and abstract" if a.strip() else "title only").rename("coded on")
    spread = pd.crosstab(per_source.rename("themes per source"), coded_on)
    print(f"{int((per_source > 0).sum())} sources use a Table 1 term, {int((per_source == 0).sum())} use none")
    display(by_theme)
    display(by_term)
    display(spread)
else:
    print("Not run yet: reference_lookups.csv is missing.")

### Run 24. Terms extracted by Qwen, factor term match and factor themes coded by Qwen

**Question.** Do factor themes coded by Qwen differ from a factor term match, and how do both compare with terms extracted by Qwen?

The three approaches:

- **Terms extracted by Qwen:** run 22, in which Qwen3.8-27B listed the words each source uses to describe research data sharing.
- **Factor term match:** run 23, in which the notebook checks each factor term in the screening guide against each source's title and abstract, with no model.
- **Factor themes coded by Qwen:** the coding in the search development, in which Qwen3.5-122B judged which of the screening guide's five factor themes each source reports.

**Measured.** For each of the five themes, and for any theme, each source gets a yes or no from factor themes coded by Qwen and from factor term match. The notebook reports percentage agreement, Cohen's kappa with a 95% confidence interval, and McNemar's exact test, for all 102 sources and for the development and test sets. It then sets the three approaches side by side on one measure, whether each reaches a source: a term extracted by Qwen, a factor term, or any factor theme coded by Qwen. For the three together it reports Fleiss' kappa and Cochran's Q, and for each pair the same statistics as above.

**Result.** Factor themes coded by Qwen and factor term match agree on 63% to 90% of sources per theme, with kappa from 0.34 to 0.64. Qwen codes more themes than factor term match finds. For factors that obstruct and factors that facilitate, factor term match finds no theme that Qwen misses, while Qwen codes factors that obstruct in 19 sources and factors that facilitate in 38 sources with no factor term for them (McNemar p < 0.001 for both). For policies and what drive them, Qwen alone codes 14 sources and factor term match alone finds 4 (p = 0.031). For stakeholder perspectives and mechanisms for implementation, the disagreements run both ways and McNemar's test shows no lean (p = 0.34 and 0.33). For any theme, Qwen codes a factor in 88 sources and factor term match finds one in 79; the two agree on 81 sources (79%), with kappa 0.32.

Terms extracted by Qwen reach 84 sources, factor term match 79 and factor themes coded by Qwen 88. All three reach 67 sources and none reaches 4; the other 31 split. Fleiss' kappa for the three is 0.31, and Cochran's Q (3.9, p = 0.14) shows no difference in how many sources each reaches. Pairwise, kappa is 0.48 for terms extracted by Qwen and factor themes coded by Qwen, 0.32 for factor term match and factor themes coded by Qwen, and 0.18 for terms extracted by Qwen and factor term match. Terms extracted by Qwen name sharing and the other two name factors, so the three agree on which sources concern the topic, not on what the sources say about it. The comparisons show where the approaches differ, not which is right.

Files: [reference_sources.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/processed/search-development/reference_sources.csv), [reference_codes_qwen3.5-122b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/reference_codes_qwen3.5-122b.csv), [terms_qwen3.8-27b.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/terms_qwen3.8-27b.csv).

In [ ]:
def agreement(a, b, a_name, b_name):
    """Percentage agreement, Cohen's kappa with a 95% confidence interval, and McNemar's exact test
    for two yes-or-no codings of the same sources."""
    n = len(a)
    yy, yn, ny, nn = int((a & b).sum()), int((a & ~b).sum()), int((~a & b).sum()), int((~a & ~b).sum())
    po = (yy + nn) / n
    pe = ((yy + yn) * (yy + ny) + (ny + nn) * (yn + nn)) / n ** 2
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    se = math.sqrt(po * (1 - po) / (n * (1 - pe) ** 2)) if pe < 1 else float("nan")
    d = yn + ny
    p = min(1.0, 2 * sum(math.comb(d, k) for k in range(min(yn, ny) + 1)) / 2 ** d) if d else 1.0
    return {"n": n, "both": yy, f"{a_name} only": yn, f"{b_name} only": ny, "neither": nn, **dict( agreement=round(po, 2),
                kappa=round(kappa, 2), kappa_low=round(max(-1.0, kappa - 1.96 * se), 2), kappa_high=round(min(1.0, kappa + 1.96 * se), 2),
                mcnemar_p=round(p, 3))}


if path("reference_lookups.csv").exists():
    coded = sources["themes"].reindex(themes.index).map(lambda s: {t.strip() for t in s.split(";") if t.strip()})
    qwen = pd.DataFrame({theme: coded.map(lambda s, t=theme: t in s) for theme in TABLE1})
    qwen["any theme"] = qwen.any(axis=1)
    keywords = themes[list(TABLE1)].copy()
    keywords["any theme"] = keywords.any(axis=1)
    compared = pd.DataFrame([dict(theme=t, set=name, **agreement(qwen.loc[rows, t], keywords.loc[rows, t], "Factor themes coded by Qwen", "Factor term match"))
                             for t in qwen.columns
                             for name, rows in (("all", qwen.index), ("development", sets == "development"),
                                                ("test", sets == "test"))]).set_index(["theme", "set"])
    display(compared)
else:
    print("Not run yet: reference_lookups.csv is missing.")

def fleiss_kappa(frame):
    """Fleiss' kappa for several yes-or-no codings of the same sources."""
    k, n = frame.shape[1], len(frame)
    yes = frame.sum(axis=1)
    p_yes = yes.sum() / (n * k)
    p_i = (yes * (yes - 1) + (k - yes) * (k - yes - 1)) / (k * (k - 1))
    pe = p_yes ** 2 + (1 - p_yes) ** 2
    return (p_i.mean() - pe) / (1 - pe)


def cochran_q(frame):
    """Cochran's Q for several yes-or-no codings of the same sources, with its p-value."""
    k = frame.shape[1]
    cols, rows = frame.sum(axis=0), frame.sum(axis=1)
    q = (k - 1) * (k * (cols ** 2).sum() - cols.sum() ** 2) / (k * rows.sum() - (rows ** 2).sum())
    # chi-squared with k - 1 degrees of freedom; for k = 3 the survival function is exp(-q / 2)
    return q, math.exp(-q / 2) if k == 3 else float("nan")


if path("reference_lookups.csv").exists() and path("terms_qwen3.8-27b.csv").exists():
    term_rows = read("terms_qwen3.8-27b.csv")
    with_terms = set(term_rows.loc[term_rows["location"] != "not in text", "ref_id"])
    three = pd.DataFrame({"Terms extracted by Qwen": qwen.index.isin(with_terms),
                          "Factor term match": keywords["any theme"].to_numpy(),
                          "Factor themes coded by Qwen": qwen["any theme"].to_numpy()}, index=qwen.index)
    patterns = three.value_counts().rename("sources").reset_index()
    pairs = [pd.DataFrame([agreement(three[a], three[b], a, b)], index=[f"{a} and {b}"])
             for a, b in (('Terms extracted by Qwen', 'Factor term match'), ('Terms extracted by Qwen', 'Factor themes coded by Qwen'), ('Factor term match', 'Factor themes coded by Qwen'))]
    q, p = cochran_q(three.astype(int))
    print(f"sources each approach reaches: " + ", ".join(f"{c} {int(three[c].sum())}" for c in three))
    print(f"Fleiss' kappa {fleiss_kappa(three.astype(int)):.2f}; Cochran's Q {q:.1f}, p = {p:.3f}")
    display(patterns)
    for pair in pairs:
        display(pair)

## Revising the query

Runs 25 to 30 revise the query on two points: they search titles and abstracts in every part of the query, and they test a research term. Run 31 measures the revised query against the test set. Every search below is named by its terms. *share near data* is a form of *share* within 5 words of *data*. The NOT clauses are named by their words.

### Run 25. Research terms and health terms, with the NOT applied to titles

**Question.** Does a research term cut the records of share near data without losing known sources, and do health terms find sources share near data misses?

**Measured.** In each database, records for 2015 to 2025 and the development and test sources found, for:

- share near data in titles, and in titles and abstracts;
- four research searches in titles and abstracts: share near *"research data"*; share near data AND *research\**; share near data AND (*research\** OR *scientific* OR *scientist\** OR *study* OR *studies* OR *trial\** OR *cohort\**); share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**);
- each search with and without NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*), with the NOT applied to titles;
- three health terms in titles and abstracts, each alone: *"individual participant data"*, *"individual patient data"* and *"clinical trial data"*, with the records each adds to share near data and the sources it finds that share near data misses.

Sources lost count against share near data in titles and abstracts with the NOT.

**Result.** With the NOT; each cell gives records, development sources found and test sources found.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data, titles | 2,436 · 20 · 8 | 7,070 · 27 · 16 | 6,149 · 28 · 12 |
| share near data, titles and abstracts | 18,255 · 21 · 9 | 70,281 · 30 · 20 | 54,521 · 31 · 14 |
| share near *"research data"* | 1,510 · 9 · 6 | 715 · 3 · 6 | 577 · 3 · 5 |
| share near data AND *research\** | 8,380 · 17 · 8 | 26,348 · 24 · 18 | 19,465 · 25 · 12 |
| share near data AND (*research\** … *study* OR *studies* … *cohort\**) | 14,050 · 20 · 9 | 43,993 · 29 · 19 | 34,292 · 29 · 13 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**) | 10,256 · 20 · 9 | 29,744 · 28 · 19 | 22,392 · 29 · 13 |

Against share near data in titles and abstracts, share near *"research data"* loses 15, 41 and 37 sources; share near data AND *research\** loses 5, 8 and 8; the two word lists lose 1 to 3. None of the health terms finds a source share near data misses, and each adds 2,180 to 5,056 records.

**Decision.** The health terms leave. Run 26 repeats the research searches with the NOT applied to titles and abstracts.

Files: [term_table_research.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_research.csv), [term_table_research_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/term_table_research_console.txt).

In [ ]:
#| eval: false
# Research and health terms, with NOT (blockchain* ... cloud) applied to titles. Writes term_table_research.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
CONTEXT = ["research*", "scientific", "scientist*", "study", "studies", "trial*", "cohort*"]
CONTEXT_NO_STUDY = [w for w in CONTEXT if w not in ("study", "studies")]
HEALTH = ['"individual participant data"', '"individual patient data"', '"clinical trial data"']
TEST = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def research_near(db, where):
    """A form of share within 5 words of "research data"."""
    if db == "PubMed":
        tag = "ti" if where == "title" else "tiab"
        return "(" + " OR ".join(f'"{w} research data"[{tag}:~5]' for w in FORMS) + ")"
    t = {"Scopus": {"title": "TITLE", "anywhere": "TITLE-ABS-KEY"},
         "Web of Science": {"title": "TI=", "anywhere": "TS="}}[db][where]
    return f'{t}(shar* W/5 "research data")' if db == "Scopus" else f'{t}(shar* NEAR/5 "research data")'


def both(a, b):
    return f"({a} AND {b})"


def minus(db, a, b):
    """Records a finds that b does not."""
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


def with_not(db, q):
    return minus(db, q, field(db, "title", NOT_TERMS))


def queries(db):
    title, anywhere = share_near_data(db, "title", 5), share_near_data(db, "anywhere", 5)
    return [("title", "baseline: share near data", title),
            ("anywhere", "baseline: share near data", anywhere),
            ("anywhere", 'research 1: share near "research data"', research_near(db, "anywhere")),
            ("anywhere", "research 2: share near data AND research*", both(anywhere, field(db, "anywhere", ["research*"]))),
            ("anywhere", "research 3: share near data AND a research word", both(anywhere, field(db, "anywhere", CONTEXT))),
            ("anywhere", "research 4: research 3 without study, studies", both(anywhere, field(db, "anywhere", CONTEXT_NO_STUDY)))]


def row(**kw):
    base = dict(database="", field="", test="", version="", NOT="", query="", records_2015_2025="",
                records_added_beyond_baseline="", development_found="", development_of="", test_found="", test_of="",
                lost_from_baseline="", beyond_baseline="")
    base.update(kw)
    return base


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    baseline = {}
    for where, name, q0 in queries(db):
        for nt in ("with NOT", "without NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name.startswith("baseline") and nt == "with NOT":
                baseline[where] = (q, d | t)
            lost = baseline["anywhere"][1] - d - t if not name.startswith("baseline") else set()
            rows.append(row(database=db, field=where, test="research", version=name, NOT=nt, query=q,
                            records_2015_2025=recs, development_found=len(d), development_of=len(dev),
                            test_found=len(t), test_of=len(test),
                            lost_from_baseline="; ".join(sorted(LABEL.get(r, r) for r in lost))))
    base_q, base_found = baseline["anywhere"]
    for term in HEALTH:
        q = field(db, "anywhere", [term])
        recs = sl.count(db, f"({q}) AND {limit_near(db)}")
        added = sl.count(db, f"({minus(db, q, base_q)}) AND {limit_near(db)}")
        d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
        beyond = (d | t) - base_found
        rows.append(row(database=db, field="anywhere", test="health", version=term, query=q, records_2015_2025=recs,
                        records_added_beyond_baseline=added, development_found=len(d), development_of=len(dev),
                        test_found=len(t), test_of=len(test),
                        beyond_baseline="; ".join(sorted(LABEL.get(r, r) for r in beyond))))
    sl.write("term_table_research.csv", rows)

### Run 26. Research terms, with the NOT applied to titles and abstracts

**Question.** With every part of the query in titles and abstracts, which research term cuts the most records for the fewest sources lost?

**Measured.** The five searches of run 25 in titles and abstracts, each with and without NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*), now applied to titles and abstracts. Sources lost count against share near data with the same NOT setting.

**Result.** With the NOT; each cell gives records, the share of records cut from share near data, and sources lost.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data | 17,790 | 62,769 | 49,700 |
| share near *"research data"* | 1,469 · 92% · 15 | 686 · 99% · 41 | 553 · 99% · 37 |
| share near data AND *research\** | 8,151 · 54% · 5 | 24,191 · 61% · 8 | 18,216 · 63% · 8 |
| share near data AND (*research\** … *study* OR *studies* … *cohort\**) | 13,741 · 23% · 1 | 40,641 · 35% · 2 | 32,259 · 35% · 3 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**) | 10,003 · 44% · 1 | 27,420 · 56% · 3 | 21,041 · 58% · 3 |

The NOT loses no source in any search. On share near data it removes 6% of records in PubMed, 24% in Scopus and 20% in Web of Science.

**Decision.** Run 27 tests *scienti\** as a shorter form of *scientific* and *scientist\**, without *trial\** and *cohort\**.

Files: [research_terms.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms.csv), [research_terms_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_console.txt).

In [ ]:
#| eval: false
# Research terms, each with and without NOT (blockchain* ... cloud) in titles and abstracts. Writes research_terms.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
WORDS = ["research*", "scientific", "scientist*", "study", "studies", "trial*", "cohort*"]
WORDS_NO_STUDY = [w for w in WORDS if w not in ("study", "studies")]
TEST = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def research_near(db):
    """A form of share within 5 words of "research data", in titles and abstracts."""
    if db == "PubMed":
        return "(" + " OR ".join(f'"{w} research data"[tiab:~5]' for w in FORMS) + ")"
    return 'TITLE-ABS-KEY(shar* W/5 "research data")' if db == "Scopus" else 'TS=(shar* NEAR/5 "research data")'


def with_not(db, q):
    """Leaves out records that name a computing word in the title or abstract."""
    t = field(db, "anywhere", NOT_TERMS)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


def searches(db):
    base = share_near_data(db, "anywhere", 5)
    return [("share near data", base),
            ('share near "research data"', research_near(db)),
            ("share near data AND research*", f"({base} AND {field(db, 'anywhere', ['research*'])})"),
            ("share near data AND a research word", f"({base} AND {field(db, 'anywhere', WORDS)})"),
            ("share near data AND a research word without study and studies",
             f"({base} AND {field(db, 'anywhere', WORDS_NO_STUDY)})")]


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    reference = {}
    for name, q0 in searches(db):
        for nt in ("without NOT", "with NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name == "share near data":
                reference[nt] = d | t
            lost = reference[nt] - d - t
            rows.append(dict(database=db, search=name, NOT=nt, query=q, records_2015_2025=recs,
                             development_found=len(d), development_of=len(dev), test_found=len(t), test_of=len(test),
                             lost_against_share_near_data="; ".join(sorted(LABEL.get(r, r) for r in lost))))
    sl.write("research_terms.csv", rows)

### Run 27. share near data AND (research\* OR scienti\*)

**Question.** Does *scienti\**, which matches scientific, scientist and scientists but not science, recover the sources *research\** alone loses, and what do *trial\** and *cohort\** add?

**Measured.** share near data AND (*research\** OR *scienti\**), in titles and abstracts, with and without NOT (*blockchain\** … *cloud*) applied to titles and abstracts, with the sources lost against share near data by name.

**Result.** With the NOT; each cell gives records and sources lost.

| Search | PubMed | Scopus | Web of Science |
|:--|:--|:--|:--|
| share near data AND *research\** | 8,151 · 5 | 24,191 · 8 | 18,216 · 8 |
| share near data AND (*research\** OR *scienti\**) | 8,706 · 3 | 25,745 · 3 | 19,501 · 5 |
| share near data AND (*research\** OR *scientific* OR *scientist\** OR *trial\** OR *cohort\**) | 10,003 · 1 | 27,420 · 3 | 21,041 · 3 |

share near data AND (*research\** OR *scienti\**) loses Mennes et al. (2013) in every database, Houtkoop et al. (2018) and Huang et al. (2013) in Scopus and Web of Science, and Mbuagbaw et al. (2017) and Savage and Vickers (2009) in PubMed and Web of Science. *trial\** and *cohort\** recover only Mbuagbaw et al. (2017) and Savage and Vickers (2009), and Scopus finds both without them, so across the three databases *trial\** and *cohort\** add no source for about 4,500 records.

**Decision.** The research term is (*research\** OR *scienti\**).

Files: [research_terms_scienti.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti.csv), [research_terms_scienti_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/research_terms_scienti_console.txt).

In [ ]:
#| eval: false
# share near data AND (research* OR scienti*), with and without NOT (blockchain* ... cloud) in titles and abstracts.
# Writes research_terms_scienti.csv.
NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
TEST = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]


def with_not(db, q):
    """Leaves out records that name a computing word in the title or abstract."""
    t = field(db, "anywhere", NOT_TERMS)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


def searches(db):
    base = share_near_data(db, "anywhere", 5)
    return [("share near data", base),
            ("share near data AND (research* OR scienti*)",
             f"({base} AND {field(db, 'anywhere', ['research*', 'scienti*'])})")]


rows = []
for db in DATABASES:
    dev = [s for s in DEV if sl.testable(db, s)]
    test = [s for s in TEST if sl.testable(db, s)]
    reference = {}
    for name, q0 in searches(db):
        for nt in ("without NOT", "with NOT"):
            q = with_not(db, q0) if nt == "with NOT" else q0
            recs = sl.count(db, f"({q}) AND {limit_near(db)}")
            d, t = sl.retrieved(db, q, dev), sl.retrieved(db, q, test)
            if name == "share near data":
                reference[nt] = d | t
            lost = sorted(LABEL.get(r, r) for r in reference[nt] - d - t)
            rows.append(dict(database=db, search=name, NOT=nt, query=q, records_2015_2025=recs,
                             development_found=len(d), development_of=len(dev), test_found=len(t), test_of=len(test),
                             lost_against_share_near_data="; ".join(lost)))
    sl.write("research_terms_scienti.csv", rows)

### Run 28. The records the NOT removes

**Question.** What records does NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*) remove from share near data AND (*research\** OR *scienti\**), in titles and abstracts?

**Measured.** The records each word matches, every record the NOT removes, and 20 of them per database drawn at random with seed 20261123.

**Result.** The NOT removes 488 records in PubMed, 5,938 in Scopus and 3,262 in Web of Science. *cloud* and *blockchain\** match the most. Of the 40 sampled titles from Scopus and Web of Science, 39 describe computing systems. Of the 20 from PubMed, most describe systems, and three concern research data sharing itself: "National and Transnational Security Implications of Asymmetric Access to and Use of Biological Data", "Connecting data, tools and people across Europe: ELIXIR's response to the COVID-19 pandemic" and a systematic review of challenges in building a digital health data integration ecosystem.

**Decision.** Run 29 tests the NOT without *cloud*, the word most likely to name research infrastructure.

Files: [not_removed_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_removed_records.csv), [not_removed_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_removed_console.txt).

In [ ]:
#| eval: false
# The records NOT (blockchain* ... cloud) removes from share near data AND (research* OR scienti*), in titles and
# abstracts: the count for each word, every removed record, and 20 at random per database. Writes not_removed_records.csv.
import random

NOT_TERMS = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
random.seed(20261123)

rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    for term in NOT_TERMS:
        print(db, term, sl.count(db, f"({q} AND {field(db, 'anywhere', [term])}) AND {lim}"))
    removed = fetch(db, f"({q} AND {field(db, 'anywhere', NOT_TERMS)})", lim)
    rows += [dict(database=db, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in removed]
    for r in random.sample(removed, min(20, len(removed))):
        print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("not_removed_records.csv", rows)

### Run 29. The NOT without cloud

**Question.** How much of the NOT's work does *cloud* do, and what do the records only *cloud* removes concern?

**Measured.** On share near data AND (*research\** OR *scienti\**), in titles and abstracts, the records removed by NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud*), by the same NOT without *cloud*, and by NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"*); and the records that name *cloud* and none of the other six words, with 20 per database drawn at random.

**Result.**

| NOT | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* OR *cloud* | 488 | 5,938 | 3,262 |
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* OR *vehicular* OR *"edge computing"* | 285 | 4,274 | 2,258 |
| *blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"* | 276 | 4,033 | 2,118 |

*cloud* alone removes 203 records in PubMed, 1,664 in Scopus and 1,004 in Web of Science. The PubMed sample includes "Population Neuroscience: Strategies to Promote Data Sharing While Protecting Privacy" and "Share all SARS-CoV-2 data immediately", and the Scopus sample "Towards a European health research and innovation cloud (HRIC)". *vehicular* and *"edge computing"* remove 9, 241 and 140 records beyond the other four words.

**Decision.** *cloud*, *vehicular* and *"edge computing"* leave the NOT. Run 30 measures each of the four words left.

Files: [cloud_only_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/cloud_only_records.csv), [not_narrowed_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_narrowed_console.txt).

In [ ]:
#| eval: false
# Three NOT clauses on share near data AND (research* OR scienti*), in titles and abstracts, and the records only cloud
# removes, with 20 at random per database. Writes cloud_only_records.csv.
import random

SEVEN = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"', "vehicular", '"edge computing"', "cloud"]
SIX = [t for t in SEVEN if t != "cloud"]
FOUR = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"']
random.seed(20261123)


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    total = sl.count(db, f"({q}) AND {lim}")
    for terms in (SEVEN, SIX, FOUR):
        print(db, len(terms), total - sl.count(db, f"({minus(db, q, field(db, 'anywhere', terms))}) AND {lim}"))
    cloud_only = fetch(db, minus(db, f"({q} AND {field(db, 'anywhere', ['cloud'])})", field(db, "anywhere", SIX)), lim)
    rows += [dict(database=db, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in cloud_only]
    for r in random.sample(cloud_only, min(20, len(cloud_only))):
        print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("cloud_only_records.csv", rows)

### Run 30. What each of blockchain\*, "smart contract\*", IoT and "internet of things" removes

**Question.** Does each of the four words remove records the others do not, and what are those records?

**Measured.** On share near data AND (*research\** OR *scienti\**), in titles and abstracts, the records each word matches that none of the other three does, with 10 per word and database drawn at random.

**Result.**

| Word | PubMed | Scopus | Web of Science |
|:--|--:|--:|--:|
| *blockchain\** | 129 | 1,401 | 706 |
| *"smart contract\*"* | 0 | 19 | 8 |
| *IoT* | 10 | 193 | 125 |
| *"internet of things"* | 17 | 301 | 166 |

*"smart contract\*"* removes almost nothing the other words do not. *blockchain\** removes "Role of Incentives in the Use of Blockchain-Based Platforms for Sharing Sensitive Health Data: Experimental Study" in PubMed and Scopus, a study of why people use a sharing system, which the eligibility criteria admit. *"internet of things"* in abstracts removes commentaries on health data such as "Susan G. Komen Big Data for Breast Cancer Initiative: How Patient Advocacy Organizations Can Facilitate Using Big Data to Improve Patient Outcomes" and "Digital technologies: tensions in privacy and data". The NOT loses no known source, but in titles and abstracts each of its words also removes eligible records, which screening could include, while the records it keeps out cost only screening.

**Decision.** The query drops the NOT. Without it, share near data AND (*research\** OR *scienti\**) returns 9,194 records in PubMed, 31,683 in Scopus and 22,763 in Web of Science, 6,427 more across the three databases than with NOT (*blockchain\** OR *"smart contract\*"* OR *IoT* OR *"internet of things"*), and loses no known source.

Files: [not_unique_records.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_unique_records.csv), [not_unique_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/not_unique_console.txt).

In [ ]:
#| eval: false
# The records each of blockchain*, "smart contract*", IoT and "internet of things" removes that none of the other three
# does, from share near data AND (research* OR scienti*), in titles and abstracts, with 10 at random. Writes
# not_unique_records.csv.
import random

FOUR = ["blockchain*", '"smart contract*"', "IoT", '"internet of things"']
random.seed(20261123)


def minus(db, a, b):
    return f"({a} AND NOT {b})" if db == "Scopus" else f"({a} NOT {b})"


rows = []
for db in DATABASES:
    q = f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"
    lim = limit_near(db)
    for term in FOUR:
        others = [t for t in FOUR if t != term]
        only = minus(db, f"({q} AND {field(db, 'anywhere', [term])})", field(db, "anywhere", others))
        n = sl.count(db, f"({only}) AND {lim}")
        print(db, term, n)
        if n:
            recs = fetch(db, only, lim)
            rows += [dict(database=db, word=term, id=r["id"], doi=r["doi"], year=r["year"], title=r["title"]) for r in recs]
            for r in random.sample(recs, min(10, len(recs))):
                print(f"    {r['year']}  {r['title'][:150]}")
    sl.write("not_unique_records.csv", rows)

### Run 31. The revised query against the test set

**Question.** How much of the held-out test set does the revised query find?

**Measured.** share within 5 words of data AND (*research\** OR *scienti\**), in titles and abstracts, against the test sources, checked by DOI or PMID in each database, with a Wilson 95% interval, as in run 16.

**Result.**

| | Found | 95% interval |
|:--|--:|:--|
| PubMed | 8 of 11 | 0.43 to 0.90 |
| Scopus | 19 of 23 | 0.63 to 0.93 |
| Web of Science | 12 of 18 | 0.44 to 0.84 |
| At least one database | 20 of 25 | 0.61 to 0.91 |

The revised query finds 20 of the 25 indexed test sources, against 18 for the query in run 16. It misses 11 test sources. Six cannot be found by any query: Bishoff and Johnston (2015), He (2016) and Zvyagintseva (2015) are in none of the databases, and Brown et al. (2015), Guédon (2015) and Koopman (2015) have no DOI or PMID. Five are indexed and not found: Chiware and Mathe (2016), Specht et al. (2015) and Wittenberg and Elings (2017) concern research data management, Watson (2015) concerns open science, and Houtkoop et al. (2018) is one of the sources the research term loses (run 27).

Files: [test_coverage_revised.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage_revised.csv), [test_missed_revised.csv](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_missed_revised.csv), [test_coverage_revised_console.txt](https://github.com/zackbatist/RDS-Review/blob/main/data/raw/search-development/test_coverage_revised_console.txt).

In [ ]:
#| eval: false
# The revised query against the test set, as in run 16. Writes test_coverage_revised.csv and test_missed_revised.csv.
def revised(db):
    """share within 5 words of data AND (research* OR scienti*), in titles and abstracts."""
    return f"({share_near_data(db, 'anywhere', 5)} AND {field(db, 'anywhere', ['research*', 'scienti*'])})"


test = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in test if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, revised(db), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=revised(db), found=len(found[db]), indexed=len(indexed[db]),
                     coverage=p, low=lo, high=hi))
any_found, in_any = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(any_found), len(in_any))
rows.append(dict(database="all three", found=len(any_found), indexed=len(in_any), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(test, key=lambda s: s["label"]):
    if s["ref_id"] in any_found:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], in_scope=s["in_scope"],
                       pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage_revised.csv", rows)
sl.write("test_missed_revised.csv", missed)

### The revised query

A form of *share* within 5 words of *data* AND (*research\** OR *scienti\**), in titles and abstracts, with no NOT clause, 2015 to 2025, English or French. It returns 9,194 records in PubMed, 31,683 in Scopus and 22,763 in Web of Science, before deduplication.

| Database | Query |
|:--|:--|
| PubMed | ("sharing data"[tiab:~5] OR "shared data"[tiab:~5] OR "share data"[tiab:~5] OR "shares data"[tiab:~5]) AND (research*[tiab] OR scienti*[tiab]) |
| Scopus | TITLE-ABS-KEY(shar* W/5 data) AND (TITLE-ABS-KEY(research*) OR TITLE-ABS-KEY(scienti*)) |
| Web of Science | TS=(shar* NEAR/5 data) AND (TS=(research*) OR TS=(scienti*)) |

## Reproducibility

The work runs in two places. The scripts in `retrieval/` and the runs call the database APIs from the analysis server. The local model runs on the GPU server: `scripts/launch_rds_llm.sh` downloads its GGUF file from Hugging Face at a fixed revision, checks its SHA-256, and starts a llama.cpp server (`scripts/README.md`). Files the APIs and the local model return go in `data/raw/search-development/`, and files this notebook builds go in `data/processed/search-development/`. Before anything else, the notebook builds `reference_sources.csv`, `counts.csv`, `verdict.csv`, `statement_matches.csv`, `terms.csv` and `term_syntax.csv` from the raw files by fixed rules, and rebuilds them each time it runs.

Every file comes from code in the repository except one judgment, the comparison of Chawinga and Zinn's criteria with RDS-Review's in `comparison.csv`. The local model runs at temperature 0 with a fixed seed and thinking off, and writes the model file, the llama.cpp build and the settings beside its output. The databases keep indexing new records, so a rerun of the searches months later returns somewhat different counts; the committed files and `log.md` record what each search returned and when.

The searches need three API keys, all registered under zachary.batist@mcgill.ca: an NCBI API key (ncbi.nlm.nih.gov, Account Settings), a Scopus API key (dev.elsevier.com), which has full Scopus Search access only from McGill's network, and a Web of Science Starter API key (developer.clarivate.com) on the Free Institutional Member Plan. Rebuilding the reference set also needs Elsevier full-text entitlement for the journal.

### Commands

Every command runs on the analysis server from a terminal. Run the long ones inside `tmux` (`tmux new -s search`, then paste; detach with Ctrl-b d, reattach with `tmux attach -t search`), so a dropped connection doesn't stop them. The commands assume the repo is cloned at `~/RDS-Review`.

1. **Keys file.** Prompts for each key without echoing it and writes `~/.rds-review-keys` outside the repo, readable by you only.

   ```bash
   umask 077
   read -rsp "NCBI API key: " NCBI; echo
   read -rsp "Scopus API key: " SCOPUS; echo
   read -rsp "Web of Science Starter API key: " WOS; echo
   printf 'export NCBI_API_KEY=%s\nexport SCOPUS_API_KEY=%s\nexport WOS_API_KEY=%s\n' "$NCBI" "$SCOPUS" "$WOS" > ~/.rds-review-keys
   chmod 600 ~/.rds-review-keys
   unset NCBI SCOPUS WOS
   grep -q rds-review-keys ~/.bashrc || echo 'source ~/.rds-review-keys' >> ~/.bashrc
   ls -l ~/.rds-review-keys
   ```

2. **Python environment.**

   ```bash
   cd ~/RDS-Review
   git pull
   python3 -m venv ~/.venvs/rds-review
   source ~/.venvs/rds-review/bin/activate
   pip install --upgrade pip
   pip install -r requirements.txt
   ```

3. **Each script.** Every script runs from the repo root. Paste the block below, with its last line swapped for the step's command from the table.

   ```bash
   cd ~/RDS-Review
   git pull
   source ~/.venvs/rds-review/bin/activate
   source ~/.rds-review-keys
   R=analysis/00_search-development/retrieval
   O=data/raw/search-development
   python3 $R/api_test.py 2>&1 | tee $O/api_test_console.txt
   ```

   In order:

   | Step | Command |
   | --- | --- |
   | API test | `python3 $R/api_test.py 2>&1 \| tee $O/api_test_console.txt` |
   | Reference set | `python3 $R/reference_set.py 2>&1 \| tee $O/reference_set_console.txt` |
   | Lookups | `python3 $R/lookups.py 2>&1 \| tee $O/lookups_console.txt` |
   | Coding, after the local model is running | `LLM_URL=http://<GPU server>:11450 python3 analysis/00_search-development/local-model/coding.py reference_codes_qwen3.5-122b.csv 2>&1 \| tee $O/coding_console_qwen3.5-122b.txt` |
   | Assignment of indexing terms, after the local model is running | `LLM_URL=http://<GPU server>:11450 python3 analysis/00_search-development/local-model/indexing.py term_assignments_qwen3.5-122b.csv 2>&1 \| tee $O/indexing_console_qwen3.5-122b.txt` |
   | Terms the sources use, after the local model is running | `LLM_URL=http://<GPU server>:11451 python3 analysis/00_search-development/local-model/terms.py terms_qwen3.8-27b.csv 2>&1 \| tee $O/terms_console_qwen3.8-27b.txt` |

   Notebook 00 runs after the lookups, to build the processed files. The runs are the code cells under Choosing the query, Testing the query and What the query misses: start Python from the repo root with the keys loaded, run the notebook's first code cell and the setup cell under Choosing the query, then the run's cell.